# DressMe · Objectif 1 — notebook Colab final (Drive)
**Colab uniquement, aucune installation sur ton PC.** Tout le code du projet est écrit dans Drive par ce notebook ; tes données restent dans `MyDrive/dressme_obj1/data/`.

```
PHOTO ─► crop (SegFormer si personne · rembg si à plat) ─► Qwen2.5-VL (prompt v1) ─► contrôles (JSON, confiance, score H)
      ─► repli kNN FashionCLIP ─► RAG (5 voisins, prompt v2) ─► REVIEW ─► ItemCard v1 (JSON) + embedding .npy
```
**Ordre :** exécuter de haut en bas avec un GPU T4. La cellule 1 redémarre le runtime toute seule (normal) ; reprends ensuite à la cellule 2.

```
MyDrive/dressme_obj1/
├── recognition/ contracts/ evaluation/     ← code (écrit par ce notebook)
└── data/
    ├── database/dressme_training_ready.csv
    ├── tunisian/images/                   ← originaux
    ├── tunisian_dataset/images + labels.csv + crops/
    ├── inbox/                             ← photos à analyser
    ├── store/                             ← RÉSULTATS JSON (items/, emb/, wardrobe.json, recognition_report.json)
    └── agent_final/                       ← caches, évaluations, poids
```

## 1. Installation sûre (redémarre le runtime à la fin : normal)

In [ ]:
import os, sys, subprocess
MARK = '/content/.dressme_deps_ok'          # disparaît avec la machine Colab : l'installation est refaite à chaque nouvelle session
if os.path.exists(MARK):
    print('✅ dépendances déjà installées dans cette session')
else:
    # IMPORTANT : ne PAS mettre -U sur numpy / pandas / pillow / scipy / scikit-learn. Les mettre à jour casse Colab
    # (ImportError « cannot import name _slice from numpy._core.umath », conflits pandas 3 / pillow 13 avec les paquets Colab).
    pip = lambda *a: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=False)
    pip('transformers>=4.45', 'accelerate>=0.30', 'bitsandbytes>=0.43', 'kagglehub', 'rembg', 'onnxruntime')
    pip('pillow-heif')                      # facultatif : photos iPhone (HEIC)
    open(MARK, 'w').write('ok')
    print('✅ installation terminée. Colab va redémarrer le runtime (message « session crashed » = NORMAL).')
    print('   Ré-exécute ensuite à partir de la section 1 : cette cellule sera sautée.')
    os.kill(os.getpid(), 9)

## 2. Google Drive et chemins

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys

if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/dressme_obj1')
DATA_ROOT = PROJECT_ROOT / 'data'
DATABASE_DIR = DATA_ROOT / 'database'
TRAINING_READY = DATABASE_DIR / 'dressme_training_ready.csv'
TN_SOURCE_IMAGES = DATA_ROOT / 'tunisian' / 'images'
TN_DATASET_DIR = DATA_ROOT / 'tunisian_dataset'
TN_DATASET_IMAGES = TN_DATASET_DIR / 'images'
TN_LABELS = TN_DATASET_DIR / 'labels.csv'
STORE = DATA_ROOT / 'store'
JSON_OUTPUT = DATA_ROOT / 'training_json_results'
INBOX = DATA_ROOT / 'inbox'
REAL30 = DATA_ROOT / 'real30'
for d in ['contracts', 'recognition', 'evaluation']: (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)
for f in [DATABASE_DIR, TN_SOURCE_IMAGES, TN_DATASET_IMAGES, STORE, JSON_OUTPUT, INBOX, REAL30]: f.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT :', PROJECT_ROOT); print('DATA_ROOT    :', DATA_ROOT)

## 3. Écriture du code du projet dans Drive
Une cellule = un fichier. Ré-exécuter écrase la version précédente ; si tu modifies un fichier déjà importé, redémarre le runtime.

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/contracts/__init__.py
from .itemcard import ItemCard, Confidence, RecognitionResult, export_json_schema
from .mapping import (CATEGORY_TO_SLOT, slot_of, default_type, usage_out, gender_out, season_out,
                      warmth_formality, load_type_dictionary, SLOT_UNCONFIRMED, TRADITIONAL_SUBTYPE_TO_SLOT, SLOTS, OPEN_TOE_TYPES, WATERPROOF_TYPES)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/contracts/mapping.py
"""Table de correspondance Objectif 1  ->  ItemCard v1.

!! Tout ce qui est marqué  [A CONFIRMER]  doit être validé avec Amir (contrat) avant le gel v1.
"""
from __future__ import annotations

SLOTS = ("top", "bottom", "onepiece", "outer", "footwear", "bag", "accessory")

# 17 catégories de l'objectif 1  ->  7 emplacements communs.  None = pas d'emplacement (article non supporté).
CATEGORY_TO_SLOT: dict[str, str | None] = {
    "tshirt": "top", "shirt": "top", "top": "top", "sweater": "top",
    "jacket": "outer", "blazer": "outer", "coat": "outer",
    "dress": "onepiece",
    "skirt": "bottom", "jeans": "bottom", "pants": "bottom", "shorts": "bottom",
    "shoes": "footwear", "bag": "bag", "accessory": "accessory",
    # [A CONFIRMER] jebba / kaftan / farmla / blouza sont des pièces uniques longues -> onepiece (+ traditional=True)
    "traditional_garment": "onepiece",
    # [A CONFIRMER] sous-vêtements : hors périmètre des tenues -> aucun emplacement, pas de fiche produite
    "underwear": None,
}

# Type fin « façon Fashion Product Images » (liste de l'objectif 6)  [A CONFIRMER avec la liste exacte d'Amir].
# Sert uniquement de repli : le type fin est normalement lu dans le catalogue (voisins kNN).
CATEGORY_DEFAULT_TYPE = {
    "tshirt": "Tshirts", "shirt": "Shirts", "top": "Tops", "sweater": "Sweatshirts", "jacket": "Jackets",
    "blazer": "Blazers", "coat": "Jackets", "dress": "Dresses", "skirt": "Skirts", "jeans": "Jeans",
    "pants": "Trousers", "shorts": "Shorts", "shoes": "Casual Shoes", "bag": "Handbags",
    "accessory": "Accessories", "traditional_garment": "Traditional", "underwear": "Innerwear",
}

# warmth 0-4 / formality 0-3 : « fixés par type (dictionnaire de l'objectif 6) ».  [PLACEHOLDER]  -> remplacer par le
# dictionnaire d'Amir dès qu'il est disponible.  Valeurs par catégorie, volontairement neutres.
_WF = {"tshirt": (1, 0), "shirt": (1, 2), "top": (1, 1), "sweater": (3, 0), "jacket": (2, 1), "blazer": (2, 3),
       "coat": (4, 2), "dress": (1, 2), "skirt": (1, 1), "jeans": (2, 1), "pants": (2, 2), "shorts": (0, 0),
       "shoes": (2, 1), "bag": (0, 1), "accessory": (1, 1), "traditional_garment": (2, 3), "underwear": (1, 0)}

USAGE_OUT = {"casual": "Casual", "formal": "Formal", "sport": "Sports", "party": "Party", "traditional": "Ethnic",
             "home": "Casual", "unknown": "Casual"}
GENDER_OUT = {"male": "Men", "female": "Women", "unisex": "Unisex", "unknown": "Unisex"}   # [A CONFIRMER] valeurs exactes
SEASON_OUT = {"summer": "summer", "winter": "winter", "spring": "spring", "fall": "fall", "all_season": "all", "unknown": "all"}

OPEN_TOE_TYPES = {"sandals", "sports sandals", "flip flops", "heels"}
WATERPROOF_TYPES = {"rain jacket"}


def slot_of(category: str | None) -> str | None:
    return CATEGORY_TO_SLOT.get(category) if category else None


def default_type(category: str) -> str:
    return CATEGORY_DEFAULT_TYPE.get(category, "Accessories")


_DICT: dict | None = None


def load_type_dictionary(path: str | None = None) -> bool:
    """Charge le dictionnaire de l'objectif 6 (JSON : {"Jackets": {"warmth": 2, "formality": 1}, ...}).
    Chemin : argument, sinon variable d'environnement DRESSME_TYPE_DICT. Renvoie True si chargé."""
    global _DICT
    import json, os
    path = path or os.environ.get("DRESSME_TYPE_DICT")
    if path and os.path.exists(path):
        _DICT = json.load(open(path, encoding="utf-8")); return True
    return False


def warmth_formality(category: str, type_: str | None = None) -> tuple[int, int, bool]:
    """(warmth, formality, from_official_dictionary). Si le dictionnaire d'Amir n'est pas chargé, on renvoie un PLACEHOLDER
    neutre et False : le service ajoute alors la raison 'warmth_formality_placeholder' (rien n'est présenté comme officiel)."""
    if _DICT and type_ in _DICT:
        d = _DICT[type_]; return int(d["warmth"]), int(d["formality"]), True
    w, f = _WF.get(category, (1, 1)); return w, f, False


# Catégories dont le slot n'est PAS confirmé : la fiche est produite mais marquée 'needs_review' (revue humaine obligatoire).
SLOT_UNCONFIRMED = {"traditional_garment"}


def usage_out(v): return USAGE_OUT.get(v or "unknown", "Casual")
def gender_out(v): return GENDER_OUT.get(v or "unknown", "Unisex")
def season_out(v): return SEASON_OUT.get(v or "unknown", "all")

# Sous-types traditionnels (colonne `subtype` du jeu tunisien) -> slot.  [A CONFIRMER avec Amir et avec une personne qui connaît ces pièces]
TRADITIONAL_SUBTYPE_TO_SLOT = {"jebba": "onepiece", "blouza": "onepiece", "kaftan": "onepiece", "farmla": "outer", "fouta": "accessory", "other": "onepiece"}

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/contracts/itemcard.py
"""ItemCard v1 : le contrat commun (produit par l'objectif 1 + crop, lu par tous)."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator

Slot = Literal["top", "bottom", "onepiece", "outer", "footwear", "bag", "accessory"]
Usage = Literal["Casual", "Smart Casual", "Formal", "Sports", "Ethnic", "Party"]
Source = Literal["obj1.llm", "obj1.knn", "user_corrected"]


class Confidence(BaseModel):
    slot: float = Field(ge=0, le=1)
    colour: float = Field(ge=0, le=1)
    type: float = Field(ge=0, le=1)


class ItemCard(BaseModel):
    id: str
    name: str
    slot: Slot
    type: str                                   # type fin (liste de l'objectif 6)
    colour: list[str] = Field(min_length=1, max_length=2)   # minuscules, anglais, 2 maximum
    usage: Usage
    warmth: int = Field(ge=0, le=4)
    formality: int = Field(ge=0, le=3)
    waterproof: bool = False
    open_toe: bool = False
    gender: str = "Unisex"
    image: str
    image_crop: str                             # vêtement seul, fond blanc, 512 px
    embedding: str                              # chemin du .npy : FashionCLIP 512-d, norme 1, calculé sur image_crop
    season: str = "all"
    pattern: Optional[str] = "plain"
    confidence: Confidence
    source: Source
    price_dt: Optional[float] = None
    traditional: bool = False

    @field_validator("colour")
    @classmethod
    def _lower(cls, v): return [c.strip().lower() for c in v]


class RecognitionResult(BaseModel):
    """Ce que renvoie le service de reconnaissance (la fiche + ce qu'il faut savoir pour décider quoi en faire)."""
    status: Literal["ok", "needs_review", "unsupported"]
    card: Optional[ItemCard] = None
    category: str = "unknown"                   # catégorie fine de l'objectif 1 (17 classes)
    reasons: list[str] = []
    trace: list[str] = []
    debug: dict = {}


def export_json_schema(path: str | Path) -> dict:
    schema = ItemCard.model_json_schema()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(schema, indent=2), encoding="utf-8")
    return schema

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/__init__.py
"""Objectif 1 — reconnaissance des vêtements (service DressMe).  Imports lourds (torch) : voir service.py."""

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/config.py
from __future__ import annotations
import os
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class Config:
    # ---- chemins ----
    root: Path = field(default_factory=lambda: Path(os.environ.get("DRESSME_ROOT", "./dressme_data")))
    # ---- modèles ----
    llm_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"
    llm_4bit: bool = True                      # False + Qwen2.5-VL-3B si le GPU est limité
    embedder_id: str = "patrickjohncyh/fashion-clip"   # FashionCLIP commun. A vérifier = même checkpoint que le script de Suzana
    max_new_tokens: int = 200
    # ---- politique de l'agent (valeurs de départ, à régler sur un jeu DEV, jamais sur le jeu réel final) ----
    conf_min: float = 0.70
    knn_k: int = 10
    rag_k: int = 5
    knn_min: float = 0.60
    attr_min: float = 0.70
    h_high: float = 0.50
    # "knn"  : après un RAG insuffisant, on retombe sur le kNN s'il est sûr (comportement du notebook v1)
    # "llm"  : on garde la réponse LLM+RAG si elle est valide et confiante (à comparer : la route knn_after_rag n'a fait que 53,8 %)
    after_rag_policy: str = "knn"
    # ---- base de connaissances ----
    kb_per_cat: int = 300
    n_test_per_cat: int = 8
    dup_sim: float = 0.97
    min_side: int = 384
    analyze_outfits: bool = False              # multi-articles : désactivé (le crop partagé fournit un article par image)
    # ---- intégration ItemCard ----
    crop_size: int = 512
    review_conf: float = 0.60                  # en dessous : l'appli doit demander une confirmation à l'utilisateur
    pixel_second_colour: bool = False          # couleur secondaire par pixels : expérimental, à valider sur les photos réelles
    crop_mode: str = "auto"                    # auto | worn | flat | none   (voir recognition/crop.py)
    seg_model_id: str = "mattmdjaga/segformer_b2_clothes"
    rembg_model: str = "u2net"                 # modèle rembg. (La version récente de rembg télécharge par défaut bria-rmbg-2.0, 1 Go : licence non commerciale à ma connaissance — à vérifier)
    crop_strategy: str = "outfit"              # outfit = une fiche par pièce détectée (filtrée) · main = UNE fiche par photo (la pièce la plus grande)
    crop_min_area_main: float = 0.03           # haut / bas / robe : part minimale de la photo (en dessous : fragment, ignoré)
    crop_min_area_small: float = 0.01          # chaussures / sac / accessoires
    crop_min_side: int = 96                    # côté minimal (px) de la boîte de la pièce ; en dessous : ignorée (inexploitable)
    crop_accessories: bool = True              # chapeau, écharpe, ceinture : False = ne pas les découper
    confirm_traditional: bool = True           # True : une pièce traditionnelle reste « needs_review » tant que son slot n'est pas validé avec Amir
    tn_refs: bool = True                       # ajoute les photos tunisiennes (split train) comme références kNN/RAG
    tunisian_prompt: bool = False              # ajoute au prompt la description des pièces tunisiennes (change les résultats : mesurer)
    subtype_min_share: float = 0.5             # part minimale des voisins pour annoncer un sous-type (jebba, farmla…)
    run_tag: str = "v1"
    seed: int = 42

    @property
    def out(self) -> Path:
        p = self.root / "agent_final"; p.mkdir(parents=True, exist_ok=True); return p


CFG = Config()

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/taxonomy.py
"""Taxonomie de l'objectif 1 (17 catégories) + tables de normalisation. Reprise telle quelle du notebook."""
CATEGORIES = ["tshirt", "shirt", "top", "sweater", "jacket", "blazer", "coat", "dress", "skirt", "jeans", "pants", "shorts",
              "underwear", "shoes", "bag", "accessory", "traditional_garment"]
DEFINITIONS = {
    "tshirt": "knitted top with short or long sleeves, no buttons or only a small placket (includes polo shirts)",
    "shirt": "woven top with a full button front and a collar (dress shirts, casual shirts)",
    "top": "blouses, tank tops, camisoles, tunics, crop tops",
    "sweater": "knitted or fleece pullovers, sweatshirts, hoodies, cardigans",
    "jacket": "light outer garment with sleeves (bomber, denim, sports, rain jacket)",
    "blazer": "tailored formal jacket with lapels",
    "coat": "long or heavy outer garment for cold weather",
    "dress": "one-piece garment covering torso and legs",
    "skirt": "garment worn from the waist down, without leg separation",
    "jeans": "denim trousers",
    "pants": "non-denim trousers, chinos, track pants, leggings",
    "shorts": "short trousers ending above the knee",
    "underwear": "briefs, boxers, bras, innerwear vests",
    "shoes": "any footwear (sneakers, sandals, heels, flats, boots)",
    "bag": "handbags, backpacks, clutches, duffel bags",
    "accessory": "watch, belt, cap, scarf, sunglasses, wallet, tie, socks, jewellery",
    "traditional_garment": "traditional or ceremonial garment (jebba, farmla, blouza, kaftan, kurta)"}
COLORS = ["beige", "black", "blue", "brown", "green", "grey", "metallic", "multicolor", "orange", "pink", "purple", "red", "white", "yellow"]
GENDERS = ["male", "female", "unisex"]
SEASONS = ["summer", "winter", "spring", "fall", "all_season"]
OCCASIONS = ["casual", "formal", "sport", "party", "traditional", "home"]
COARSE = {"tshirt": "upper", "shirt": "upper", "top": "upper", "sweater": "upper", "jacket": "outer", "blazer": "outer", "coat": "outer",
          "jeans": "lower", "pants": "lower", "shorts": "lower", "skirt": "lower"}
def coarse(c): return COARSE.get(c, c)

# --- catalogue Fashion Product Images : articleType (minuscules) -> catégorie ; types ambigus exclus ---
ART = {"tshirts": "tshirt", "shirts": "shirt", "tops": "top", "tunics": "top", "sweatshirts": "sweater", "sweaters": "sweater", "jeans": "jeans",
       "trousers": "pants", "track pants": "pants", "leggings": "pants", "capris": "pants", "jeggings": "pants", "shorts": "shorts", "skirts": "skirt",
       "dresses": "dress", "jackets": "jacket", "rain jacket": "jacket", "blazers": "blazer", "casual shoes": "shoes", "sports shoes": "shoes",
       "formal shoes": "shoes", "sandals": "shoes", "sports sandals": "shoes", "heels": "shoes", "flats": "shoes", "flip flops": "shoes", "boots": "shoes",
       "handbags": "bag", "backpacks": "bag", "duffel bag": "bag", "clutches": "bag", "messenger bag": "bag", "briefs": "underwear", "bra": "underwear",
       "innerwear vests": "underwear", "boxers": "underwear", "watches": "accessory", "belts": "accessory", "caps": "accessory", "scarves": "accessory",
       "sunglasses": "accessory", "wallets": "accessory", "ties": "accessory", "socks": "accessory"}
COLOR_MAP = {"black": "black", "white": "white", "off white": "white", "cream": "beige", "beige": "beige", "nude": "beige", "skin": "beige", "khaki": "beige",
             "tan": "brown", "taupe": "brown", "brown": "brown", "coffee brown": "brown", "mushroom brown": "brown", "bronze": "brown", "copper": "brown",
             "blue": "blue", "navy blue": "blue", "turquoise blue": "blue", "teal": "blue", "grey": "grey", "grey melange": "grey", "charcoal": "grey", "steel": "grey",
             "silver": "metallic", "gold": "metallic", "metallic": "metallic", "red": "red", "maroon": "red", "burgundy": "red", "magenta": "pink", "pink": "pink",
             "rose": "pink", "peach": "pink", "mauve": "purple", "purple": "purple", "lavender": "purple", "orange": "orange", "rust": "orange", "yellow": "yellow",
             "mustard": "yellow", "green": "green", "olive": "green", "lime green": "green", "sea green": "green", "fluorescent green": "green", "multi": "multicolor"}
GENDER_MAP = {"men": "male", "women": "female", "unisex": "unisex", "boys": "male", "girls": "female"}
USAGE_MAP = {"casual": "casual", "smart casual": "casual", "travel": "casual", "ethnic": "traditional", "formal": "formal", "sports": "sport", "party": "party", "home": "home"}

# --- synonymes de sortie du LLM ---
CAT_SYN = {"t-shirt": "tshirt", "t_shirt": "tshirt", "tee": "tshirt", "polo": "tshirt", "hoodie": "sweater", "sweatshirt": "sweater", "pullover": "sweater",
           "cardigan": "sweater", "blouse": "top", "tank_top": "top", "trousers": "pants", "leggings": "pants", "sneakers": "shoes", "shoe": "shoes", "boots": "shoes",
           "sandals": "shoes", "heels": "shoes", "handbag": "bag", "backpack": "bag", "jean": "jeans", "skirts": "skirt", "dresses": "dress", "coats": "coat",
           "jackets": "jacket", "shirts": "shirt", "sweaters": "sweater", "accessories": "accessory", "traditional": "traditional_garment",
           "kaftan": "traditional_garment", "kurta": "traditional_garment"}
ATTR_SYN = {"color": {"multi_color": "multicolor", "multicolored": "multicolor", "navy": "blue", "gray": "grey", "khaki": "beige", "cream": "beige",
                      "silver": "metallic", "gold": "metallic"},
            "gender": {"men": "male", "man": "male", "boy": "male", "women": "female", "woman": "female", "girl": "female"},
            "season": {"autumn": "fall", "all_seasons": "all_season", "all": "all_season"},
            "occasion": {"sports": "sport", "smart_casual": "casual", "everyday": "casual", "ethnic": "traditional"}}
ATTR_ALLOWED = {"color": COLORS, "gender": GENDERS, "season": SEASONS, "occasion": OCCASIONS}

# --- pièces tunisiennes : sous-types (colonne `subtype` du jeu tunisien) et aide pour le prompt ---
TN_SUBTYPES = ["jebba", "farmla", "blouza", "kaftan", "fouta", "other"]
# !! Descriptions écrites de mémoire : À VALIDER / CORRIGER par quelqu'un qui connaît ces pièces (elles sont injectées dans le prompt).
TUNISIAN_HINTS = ("- Tunisian / Maghrebi garments all belong to traditional_garment: jebba (long loose tunic-like garment, sleeveless or short-sleeved), "
                  "farmla (short embroidered waistcoat or vest), blouza (long traditional dress, often embroidered), kaftan (long robe with wide sleeves), "
                  "fouta (striped or checked cotton wrap). Western vests, tunics and long dresses are NOT traditional_garment.")

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/imaging.py
from __future__ import annotations
import hashlib
import numpy as np
from PIL import Image
try:                                   # photos iPhone (HEIC) : pip install pillow-heif  (facultatif)
    import pillow_heif; pillow_heif.register_heif_opener()
except Exception: pass


def open_rgb(src) -> Image.Image:
    """Ouvre une image ; un PNG transparent est composé sur fond blanc (format du crop partagé)."""
    im = src if isinstance(src, Image.Image) else Image.open(src)
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (255, 255, 255, 255)); bg.alpha_composite(im); im = bg
    return im.convert("RGB")


def load_image(src, min_side: int = 384, max_side: int = 1024) -> Image.Image:
    im = open_rgb(src); w, h = im.size; s = min(w, h)
    if s < min_side:
        k = min_side / s; im = im.resize((round(w * k), round(h * k)), Image.LANCZOS)
    if max(im.size) > max_side: im.thumbnail((max_side, max_side), Image.LANCZOS)
    return im


def pad_square(im: Image.Image, fill=(255, 255, 255)) -> Image.Image:
    w, h = im.size; s = max(w, h); bg = Image.new("RGB", (s, s), fill); bg.paste(im, ((s - w) // 2, (s - h) // 2)); return bg


def normalise_crop(im: Image.Image, size: int = 512) -> Image.Image:
    """Format du contrat : carré, fond blanc, 512 px."""
    return pad_square(open_rgb(im)).resize((size, size), Image.LANCZOS)


def img_key(im: Image.Image) -> str:
    return hashlib.md5(im.resize((64, 64)).tobytes()).hexdigest()[:16]


def check_crop(src, min_px: int = 224) -> dict:
    """Contrôle qualité du crop reçu (tâche 2) : renvoie {ok, reasons, coverage}.  Bloquant => la fiche sera 'needs_review'."""
    im = open_rgb(src); a = np.asarray(im); mask = ~(a >= 240).all(-1); cov = float(mask.mean()); reasons = []
    if min(im.size) < min_px: reasons.append("low_resolution")
    if cov < 0.03: reasons.append("garment_too_small_or_empty")        # (un vêtement blanc sur blanc déclenche aussi ceci)
    if cov > 0.92: reasons.append("background_not_removed")
    return dict(ok=not reasons, reasons=reasons, coverage=round(cov, 3), size=im.size)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/colour.py
"""Témoin de couleur par pixels (expérimental).  La couleur est l'attribut le plus faible (72,5 % sur le catalogue ;
61-62 % dans les autres fiches). Sur un crop à fond blanc, un histogramme HSV est un second avis indépendant du LLM.
Règles écrites à la main, NON validées : ne sert qu'à nourrir la confiance (et, si activé, la couleur secondaire)."""
from __future__ import annotations
import numpy as np
from PIL import Image
from .imaging import open_rgb

NAMES = ["black", "white", "grey", "beige", "brown", "red", "orange", "yellow", "green", "blue", "purple", "pink"]


def _classify(h, s, v):
    h = h * 360.0 / 255.0; s = s / 255.0; v = v / 255.0
    conds = [v < 0.18, (s < 0.12) & (v >= 0.85), s < 0.12,
             (h >= 20) & (h < 60) & (s < 0.35) & (v >= 0.6), (h >= 8) & (h < 50) & (v < 0.6),
             (h < 15) | (h >= 345), h < 40, h < 70, h < 165, h < 255, h < 300]
    return np.select(conds, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], default=11)   # 11 = pink


def dominant_colours(src, top: int = 2) -> list[tuple[str, float]]:
    im = open_rgb(src); rgb = np.asarray(im); mask = ~(rgb >= 240).all(-1)
    if mask.mean() < 0.02: return [("white", 1.0)]
    hsv = np.asarray(im.convert("HSV"))[mask]
    idx = _classify(hsv[:, 0].astype(float), hsv[:, 1].astype(float), hsv[:, 2].astype(float))
    share = np.bincount(idx, minlength=len(NAMES)) / len(idx); order = np.argsort(-share)
    out = [(NAMES[i], float(share[i])) for i in order[:max(top, 3)] if share[i] > 0.02]
    if len(out) >= 3 and out[0][1] < 0.40 and out[2][1] >= 0.15: return [("multicolor", out[0][1])] + out[:1]
    return out[:top]

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/crop.py
"""Crop (découpe du vêtement) : photo de téléphone  ->  une ou plusieurs images « vêtement seul, fond blanc, 512 px ».

Méthode du document d'orchestration :
  * photo PORTÉE (une personne)  : SegFormer B2 (parsing de vêtements) -> un masque par pièce (haut, bas, robe, chaussures, sac, écharpe…)
  * photo À PLAT                 : rembg retire le fond
  * contrôle qualité             : masque trop petit / vêtement coupé par le cadre -> `quality.ok = False` (la fiche sera needs_review)

!! Le crop est proposé à Amir (app/segment.py).  Ce module est le REPLI : si Amir livre son code, branche-le avec
   DRESSME_CROP_HOOK='module:function'  (function(PIL.Image) -> list[(name, slot_hint, PIL.Image 512x512)]) — rien d'autre à changer.
!! Le modèle SegFormer ci-dessous est celui que je connais pour le parsing de vêtements ; son identifiant (cfg.seg_model_id) et le
   numéro des classes sont à vérifier à la première exécution (la cellule de test du notebook affiche les classes trouvées)."""
from __future__ import annotations
import importlib, os
from dataclasses import dataclass, field
import numpy as np
from PIL import Image
from .imaging import normalise_crop, open_rgb

# classes du modèle mattmdjaga/segformer_b2_clothes (ordre de sa carte de modèle ; vérifié par `describe_labels` au chargement)
SEG_LABELS = {0: "background", 1: "hat", 2: "hair", 3: "sunglasses", 4: "upper", 5: "skirt", 6: "pants", 7: "dress", 8: "belt", 9: "left_shoe",
              10: "right_shoe", 11: "face", 12: "left_leg", 13: "right_leg", 14: "left_arm", 15: "right_arm", 16: "bag", 17: "scarf"}
# (nom de la pièce, classes fusionnées, slot probable)
PIECES = [("top", [4], "top"), ("bottom", [5, 6], "bottom"), ("onepiece", [7], "onepiece"), ("shoes", [9, 10], "footwear"),
          ("bag", [16], "bag"), ("hat", [1], "accessory"), ("scarf", [17], "accessory"), ("belt", [8], "accessory")]
MAJOR = {"top", "bottom", "onepiece"}      # grandes pièces ; les autres (chaussures, sac, accessoires) sont « petites »
PERSON = [2, 11, 12, 13, 14, 15]          # cheveux, visage, jambes, bras : indices qu'une personne est dans la photo


@dataclass
class CropPiece:
    name: str
    slot_hint: str | None
    image: Image.Image
    area_frac: float = 1.0
    quality: dict = field(default_factory=lambda: dict(ok=True, reasons=[]))


class SegformerClothes:
    def __init__(self, model_id: str = "mattmdjaga/segformer_b2_clothes", device: str | None = None):
        import torch
        from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
        self.torch = torch; self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.proc = SegformerImageProcessor.from_pretrained(model_id); self.model = AutoModelForSemanticSegmentation.from_pretrained(model_id).eval().to(self.dev)
        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()}

    def describe_labels(self) -> str:
        norm = lambda v: {"upper_clothes": "upper"}.get(str(v).lower().replace("-", "_").replace(" ", "_"), str(v).lower().replace("-", "_").replace(" ", "_"))
        diff = {k: (SEG_LABELS.get(k), v) for k, v in self.id2label.items() if SEG_LABELS.get(k, "").lower() != norm(v)}
        return "classes SegFormer cohérentes avec le code" if not diff else f"⚠️ classes différentes de celles attendues (à corriger dans SEG_LABELS) : {diff}"

    def __call__(self, image: Image.Image) -> np.ndarray:
        torch = self.torch
        with torch.no_grad():
            lg = self.model(**self.proc(images=image, return_tensors="pt").to(self.dev)).logits
            up = torch.nn.functional.interpolate(lg, size=image.size[::-1], mode="bilinear", align_corners=False)
        return up.argmax(1)[0].cpu().numpy()


_SESSIONS: dict = {}


def rembg_remove(image: Image.Image, model: str = "u2net") -> Image.Image:
    """RGBA (fond transparent). Repli sans rembg : seuil sur le blanc (valable seulement pour une photo déjà sur fond clair)."""
    try:
        from rembg import remove, new_session
        if model not in _SESSIONS: _SESSIONS[model] = new_session(model)
        return remove(image, session=_SESSIONS[model]).convert("RGBA")
    except Exception as e:                                                 # rembg absent / modèle non téléchargeable
        print("⚠️ rembg indisponible (", repr(e)[:80], ") -> repli par seuil de blanc")
        a = np.asarray(image.convert("RGB")); m = ~(a >= 235).all(-1); out = image.convert("RGBA"); out.putalpha(Image.fromarray((m * 255).astype("uint8"))); return out


def piece_from_mask(image: Image.Image, mask: np.ndarray, size: int = 512, margin: float = 0.04, min_area: float = 0.01, name="item", slot_hint=None) -> CropPiece | None:
    H, W = mask.shape; area = float(mask.mean()); ys, xs = np.where(mask)
    if len(ys) == 0: return None
    reasons = []
    if area < min_area: reasons.append("mask_too_small")
    border = (mask[0].sum() + mask[-1].sum() + mask[:, 0].sum() + mask[:, -1].sum()) / max(1, mask.sum())
    if border > 0.02: reasons.append("garment_cut_by_frame")               # le vêtement touche le bord de la photo
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1; my, mx = int((y1 - y0) * margin), int((x1 - x0) * margin)
    y0, y1, x0, x1 = max(0, y0 - my), min(H, y1 + my), max(0, x0 - mx), min(W, x1 + mx)
    rgb = np.asarray(image.convert("RGB"))[y0:y1, x0:x1].copy(); m = mask[y0:y1, x0:x1]; rgb[~m] = 255             # fond blanc hors masque
    if min(y1 - y0, x1 - x0) < 64: reasons.append("crop_low_resolution")
    return CropPiece(name, slot_hint, normalise_crop(Image.fromarray(rgb), size), area, dict(ok=not reasons, reasons=reasons))


class Cropper:
    def __init__(self, size: int = 512, mode: str = "auto", seg_model_id: str = "mattmdjaga/segformer_b2_clothes", segmenter=None, remover=None,
                 hook=None, min_area: float = 0.01, rembg_model: str = "u2net", strategy: str = "outfit", min_area_main: float = 0.03,
                 min_area_small: float = 0.01, min_side: int = 96, accessories: bool = True):
        self.rembg_model, self.strategy, self.min_area_main, self.min_area_small, self.min_side, self.accessories = rembg_model, strategy, min_area_main, min_area_small, min_side, accessories
        self.size, self.mode, self.seg_model_id, self._seg, self._rem, self.hook, self.min_area = size, mode, seg_model_id, segmenter, remover, hook, min_area

    @classmethod
    def from_config(cls, cfg):
        spec = os.environ.get("DRESSME_CROP_HOOK"); hook = None
        if spec: mod, fn = spec.split(":"); hook = getattr(importlib.import_module(mod), fn)
        c = cls(cfg.crop_size, cfg.crop_mode, cfg.seg_model_id, hook=hook, rembg_model=cfg.rembg_model); c.apply_config(cfg); return c

    def apply_config(self, cfg):
        """Relit les réglages de crop dans CFG (appelé avant chaque lot : modifier CFG dans le notebook suffit)."""
        self.mode, self.strategy, self.min_area_main, self.min_area_small = cfg.crop_mode, cfg.crop_strategy, cfg.crop_min_area_main, cfg.crop_min_area_small
        self.min_side, self.accessories = cfg.crop_min_side, cfg.crop_accessories

    @property
    def segmenter(self):
        if self._seg is None: self._seg = SegformerClothes(self.seg_model_id); print(self._seg.describe_labels())
        return self._seg

    @property
    def remover(self): return self._rem or (lambda im: rembg_remove(im, self.rembg_model))

    def _flat(self, im) -> list[CropPiece]:
        rgba = np.asarray(self.remover(im).convert("RGBA")); mask = rgba[..., 3] > 127
        p = piece_from_mask(im, mask, self.size, min_area=self.min_area, name="item"); return [p] if p else []

    def _worn(self, im, labels) -> list[CropPiece]:
        """Une pièce par classe de segmentation, en IGNORANT les fragments : trop petits (part de la photo) ou boîte trop étroite (px)."""
        out = []
        for name, ids, slot in PIECES:
            if slot == "accessory" and not self.accessories: continue
            m = np.isin(labels, ids); area = float(m.mean())
            if area < (self.min_area_main if slot in MAJOR else self.min_area_small): continue
            ys, xs = np.where(m)
            if min(ys.max() - ys.min() + 1, xs.max() - xs.min() + 1) < self.min_side: continue
            p = piece_from_mask(im, m, self.size, min_area=0.0, name=name, slot_hint=slot)         # min_area=0 : le filtrage est déjà fait ci-dessus
            if p: out.append(p)
        return out

    def crop(self, photo, mode: str | None = None, strategy: str | None = None) -> list[CropPiece]:
        im = open_rgb(photo); mode = mode or self.mode; strategy = strategy or self.strategy
        if self.hook: return [CropPiece(n, s, i) for n, s, i in self.hook(im)]
        if mode == "none": return [CropPiece("whole", None, normalise_crop(im, self.size))]
        if mode == "flat": pieces = self._flat(im)
        else:
            labels = self.segmenter(im)
            if mode == "worn" or np.isin(labels, PERSON).mean() > 0.003:                                    # une personne est visible
                pieces = self._worn(im, labels) or (self._flat(im) if mode != "worn" else [])           # rien d'exploitable : on essaie rembg
            else: pieces = self._flat(im)                                                                    # sinon : photo à plat
            if strategy == "main" and len(pieces) > 1: pieces = [max(pieces, key=lambda p: p.area_frac)]  # une seule fiche par photo
        if not pieces:                                                      # rien de segmenté : on garde l'image entière, signalée
            pieces = [CropPiece("whole", None, normalise_crop(im, self.size), 1.0, dict(ok=False, reasons=["no_garment_segmented"]))]
        return pieces

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/parsing.py
from __future__ import annotations
import json, re
import numpy as np
from .taxonomy import CATEGORIES, CAT_SYN, ATTR_SYN, ATTR_ALLOWED


def _lab(v): return str(v).strip().lower().replace(" ", "_").replace("-", "_") if v is not None else "unknown"


def extract_json(text):
    if not isinstance(text, str): return None
    t = text.strip().replace("```json", "").replace("```", ""); i, j = t.find("{"), t.rfind("}")
    if i < 0 or j <= i: return None
    s = t[i:j + 1]
    for cand in (s, re.sub(r",\s*}", "}", s)):
        try: return json.loads(cand)
        except Exception: pass
    return None


def parse_answer(raw) -> dict:
    out = dict(valid=False, category="invalid", color="unknown", gender="unknown", season="unknown", occasion="unknown",
               confidence=0.0, evidence="", alternative="", issues=[], raw=raw, latency=0.0)
    o = extract_json(raw)
    if not isinstance(o, dict): out["issues"] = ["no_json"]; return out
    cat = _lab(o.get("category")); cat = CAT_SYN.get(cat, cat); issues = []
    for attr, allowed in ATTR_ALLOWED.items():
        v = _lab(o.get(attr)); v = ATTR_SYN[attr].get(v, v)
        if v in allowed or v == "unknown": out[attr] = v
        else: out[attr] = "unknown"; issues.append(f"{attr}={v}")
    try: conf = float(o.get("confidence", 0.0))
    except Exception: conf = 0.0
    if 1 < conf <= 100: conf /= 100
    alt = _lab(o.get("alternative")); alt = CAT_SYN.get(alt, alt)
    if cat not in CATEGORIES: issues.append(f"category={cat}")
    out.update(valid=cat in CATEGORIES, category=cat if cat in CATEGORIES else "invalid", confidence=float(np.clip(conf, 0, 1)),
               evidence=str(o.get("evidence", ""))[:160], alternative=alt if alt in CATEGORIES else "", issues=issues)
    return out

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/prompts.py
from .taxonomy import DEFINITIONS, COLORS, GENDERS, SEASONS, OCCASIONS, TUNISIAN_HINTS

SYSTEM = ("You are DressMe's vision analyst, a meticulous fashion cataloguer. You report only what is visible in the photo. "
          "You never invent brands, fabrics or details you cannot see. When you are unsure, you lower your confidence "
          "or answer 'unknown' for an attribute instead of guessing.")
RULES = '''- Classify the MAIN item: the garment or accessory the photo is about. If a person wears several items, choose the most prominent one.
- Polo shirts and knitted short-sleeve tops with a small collar -> tshirt.
- Hoodies, sweatshirts, pullovers, cardigans -> sweater.  Denim trousers -> jeans.  Leggings, chinos, track pants -> pants.
- Sneakers, sandals, heels, flats, boots -> shoes.  Watches, belts, caps, scarves, sunglasses, wallets, ties, socks -> accessory.
- Kaftans, jebbas, farmlas, kurtas and other ceremonial garments -> traditional_garment.
- Colour = dominant colour of the garment only (ignore skin, hair, background).
- gender / season / occasion: infer only from visible cues (cut, fabric look, context); otherwise answer "unknown".'''
CONFIDENCE_GUIDE = ("confidence in [0,1]: >0.85 clear and unambiguous; 0.6-0.85 plausible but a close alternative exists; "
                    "<0.6 item partly visible, unusual, or ambiguous.")
SCHEMA = ('{"evidence": "<=15 words describing the visible cues", "category": "...", "alternative": "second most likely category", '
          '"color": "...", "gender": "...", "season": "...", "occasion": "...", "confidence": 0.0}')


def _header(tunisian: bool = False):
    defs = "\n".join(f"- {c}: {d}" for c, d in DEFINITIONS.items())
    rules = RULES + ("\n" + TUNISIAN_HINTS if tunisian else "")
    return (f"CATEGORIES (choose exactly one):\n{defs}\n\nRULES:\n{rules}\n\n"
            f"ALLOWED VALUES:\ncolor: {COLORS}\ngender: {GENDERS}\nseason: {SEASONS}\noccasion: {OCCASIONS}\n"
            f"(for gender, season, occasion you may also answer \"unknown\")\n\n{CONFIDENCE_GUIDE}\n")


def prompt_v1(tunisian: bool = False) -> str:
    return ("TASK: analyse the clothing item in the photo.\n"
             "Work in this order, silently: (1) identify the object and its visible cues (shape, neckline, sleeves, length, closure); "
             "(2) pick the category with the definitions and rules; (3) read the colour; (4) judge gender/season/occasion only from visible cues; (5) rate your confidence.\n\n"
             + _header(tunisian) + f"\nOUTPUT: one JSON object, no markdown, no text before or after, exactly these keys:\n{SCHEMA}\n\nJSON:")


PROMPT_V1 = prompt_v1(False)


def prompt_rag(neighbors, prev, kv, flags, tunisian: bool = False):
    refs = "\n".join(f'{i}. category={n["category"]}; color={n["color"] or "?"}; gender={n["gender"] or "?"}; season={n["season"] or "?"}; '
                     f'occasion={n["occasion"] or "?"}; catalogue name="{n["name"][:70]}"; similarity={n["sim"]:.2f}' for i, n in enumerate(neighbors, 1))
    cons = ", ".join(f"{c} {s:.0%}" for c, s in list(kv["dist"].items())[:3])
    first = (f'category={prev["category"]}, color={prev["color"]}, gender={prev["gender"]}, confidence={prev["confidence"]:.2f}'
             if prev["valid"] else "your first answer was not valid or used values outside the allowed lists")
    return ("TASK: re-examine the photo and correct your first analysis if needed.\n\n"
            f"VERIFIED REFERENCES: catalogue items visually similar to the photo, with expert labels. They are evidence, not the truth: this photo may differ.\n{refs}\n"
            f"Weighted vote of the references on the category: {cons}.\n\n"
            f"YOUR FIRST ANALYSIS: {first}.\nPROBLEMS DETECTED: {', '.join(flags) if flags else 'low confidence'}.\n\n"
            "REVISION RULES:\n- Look at the photo again. If it matches the references, align with them.\n"
            "- If the photo clearly contradicts the references, keep what you see and lower your confidence.\n"
            "- Never copy an attribute from a reference that you cannot see in the photo; use \"unknown\" instead.\n\n"
            + _header(tunisian) + f"\nOUTPUT: one JSON object, no markdown, exactly these keys:\n{SCHEMA}\n\nJSON:")

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/retrieval.py
from __future__ import annotations
import numpy as np
import pandas as pd


def _s(v): return v if isinstance(v, str) else None


class Retriever:
    """kNN / RAG sur l'index FashionCLIP. Les vecteurs de l'index et de la requête doivent venir du MÊME encodeur."""
    def __init__(self, E: np.ndarray, meta: pd.DataFrame, T: float = 0.05):
        self.E, self.meta, self.T = E.astype("float32"), meta.reset_index(drop=True), T
        assert len(self.E) == len(self.meta), "index et métadonnées incohérents"

    def add(self, meta: pd.DataFrame, E: np.ndarray) -> "Retriever":
        """Nouveau Retriever = index actuel + références supplémentaires (ex. photos tunisiennes). L'encodeur doit être le même."""
        assert E.shape[1] == self.E.shape[1], "dimension d'embedding différente"
        cols = list(self.meta.columns); m = meta.copy()
        for c in cols:
            if c not in m: m[c] = None
        return Retriever(np.vstack([self.E, E.astype("float32")]), pd.concat([self.meta, m[cols]], ignore_index=True), self.T)

    def search(self, q: np.ndarray, k: int) -> list[dict]:
        s = self.E @ q; k = min(k, len(s) - 1); idx = np.argpartition(-s, k)[:k]; idx = idx[np.argsort(-s[idx])]
        w = np.exp((s[idx] - s[idx].max()) / self.T); w /= w.sum(); m = self.meta
        has = lambda c: c in m.columns
        return [dict(id=int(m.id[j]), category=m.category[j], color=_s(m.color[j]), gender=_s(m.gender[j]), season=_s(m.season[j]),
                     occasion=_s(m.occasion[j]), name=str(m.name[j]), article_type=_s(m.article_type[j]) if has("article_type") else None,
                     subtype=_s(m.subtype[j]) if has("subtype") else None,
                     image_path=str(m.image_path[j]) if has("image_path") else "", sim=float(s[j]), w=float(wi)) for j, wi in zip(idx, w)]


def knn_vote(nb):
    d = {}
    for n in nb: d[n["category"]] = d.get(n["category"], 0.0) + n["w"]
    cat = max(d, key=d.get); return dict(category=cat, conf=float(d[cat]), dist=dict(sorted(d.items(), key=lambda x: -x[1])))


def consensus(nb, attr):
    d = {}
    for n in nb:
        if n[attr]: d[n[attr]] = d.get(n[attr], 0.0) + n["w"]
    if not d: return None, 0.0
    v = max(d, key=d.get); return v, d[v] / sum(d.values())


def attr_share(nb, attr, value) -> float:
    """Part (pondérée) des voisins dont l'attribut vaut `value`."""
    tot = sum(n["w"] for n in nb if n[attr]); 
    return sum(n["w"] for n in nb if n[attr] == value) / tot if tot else 0.0

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/embedder.py
"""Encodeur FashionCLIP.  RÈGLE D'OR : l'embedding écrit dans la fiche (.npy) vient d'un FashionCLIP GELÉ (identique à celui des
objectifs 2-6).  Si l'objectif 1 fait un fine-tuning, l'encodeur affiné ne sert QU'AU kNN/RAG interne (jamais à la fiche)."""
from __future__ import annotations
import copy, importlib, os
import numpy as np


class FashionClipEmbedder:
    def __init__(self, model_id: str | None = None, model=None, proc=None, device: str | None = None):
        import torch
        from transformers import CLIPModel, CLIPProcessor
        self.torch = torch; self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if model is None:
            self.proc = CLIPProcessor.from_pretrained(model_id); self.model = CLIPModel.from_pretrained(model_id).eval().to(self.dev)
        else: self.proc, self.model = proc, model

    def images(self, imgs, bs: int = 32) -> np.ndarray:
        torch = self.torch; out = []
        with torch.no_grad():
            for i in range(0, len(imgs), bs):
                inp = {k: v.to(self.dev) for k, v in self.proc(images=imgs[i:i + bs], return_tensors="pt").items()}
                f = self.model.get_image_features(**inp); f = f.pooler_output if hasattr(f, "pooler_output") else f
                out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.vstack(out).astype("float32")

    def texts(self, prompts: list[str]) -> np.ndarray:
        """Vecteurs texte normalisés (sonde zéro-shot : FashionCLIP reconnaît-il une catégorie sans exemples ?)."""
        torch = self.torch
        with torch.no_grad():
            inp = {k: v.to(self.dev) for k, v in self.proc(text=prompts, return_tensors="pt", padding=True, truncation=True).items()}
            f = self.model.get_text_features(**inp); f = f.pooler_output if hasattr(f, "pooler_output") else f
        return torch.nn.functional.normalize(f, dim=-1).cpu().numpy().astype("float32")

    def clone(self) -> "FashionClipEmbedder":
        """Copie indépendante (pour le fine-tuning) : le modèle gelé de référence n'est jamais modifié."""
        return FashionClipEmbedder(model=copy.deepcopy(self.model), proc=self.proc, device=self.dev)


def load_shared_embed_hook(spec: str | None = None):
    """Branche le script commun de Suzana : DRESSME_SHARED_EMBED='package.module:function'.
    La fonction doit prendre une image PIL (le crop 512) et renvoyer un vecteur (512,) de norme 1. Sinon : None."""
    spec = spec or os.environ.get("DRESSME_SHARED_EMBED")
    if not spec: return None
    mod, fn = spec.split(":"); return getattr(importlib.import_module(mod), fn)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/llm.py
from __future__ import annotations
import json
from pathlib import Path
from .prompts import SYSTEM


class VisionLLM:
    def __init__(self, model_id: str, load_4bit: bool = True, max_new_tokens: int = 200):
        import torch
        from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
        self.max_new_tokens = max_new_tokens; kw = dict(device_map="auto")
        if load_4bit: kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else: kw["dtype"] = torch.float16
        try: self.model = AutoModelForImageTextToText.from_pretrained(model_id, **kw).eval()
        except (TypeError, ValueError):
            if "dtype" in kw: kw["torch_dtype"] = kw.pop("dtype")
            self.model = AutoModelForImageTextToText.from_pretrained(model_id, **kw).eval()
        self.processor = AutoProcessor.from_pretrained(model_id, min_pixels=128 * 28 * 28, max_pixels=512 * 28 * 28)
        self.torch = torch

    def generate(self, image, prompt, system=SYSTEM):
        err = None
        for use_system in (True, False):                      # certains modèles n'acceptent pas le rôle « system »
            try:
                msgs = [{"role": "system", "content": [{"type": "text", "text": system}]}] if use_system else []
                msgs.append({"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt if use_system else system + "\n\n" + prompt}]})
                text = self.processor.apply_chat_template(msgs, add_generation_prompt=True)
                inp = self.processor(text=[text], images=[image], return_tensors="pt").to(self.model.device, dtype=self.model.dtype)
                with self.torch.no_grad(): out = self.model.generate(**inp, max_new_tokens=self.max_new_tokens, do_sample=False)
                return self.processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]
            except Exception as e: err = e
        raise err


class JsonlCache:
    """Cache disque des appels LLM (génération déterministe) ; une erreur n'est jamais mise en cache."""
    def __init__(self, path):
        self.path = Path(path); self.d = {}
        if self.path.exists():
            for line in open(self.path, encoding="utf-8"):
                try:
                    r = json.loads(line)
                    if not str(r.get("raw", "")).startswith("ERROR"): self.d[r["key"]] = r
                except Exception: pass
    def __contains__(self, k): return k in self.d
    def __getitem__(self, k): return self.d[k]
    def put(self, k, **v):
        r = dict(key=k, **v); self.d[k] = r
        with open(self.path, "a", encoding="utf-8") as f: f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/knowledge_base.py
"""Base kNN/RAG construite automatiquement depuis Fashion Product Images (aucune étiquette manuelle).
Nouveauté v2 : on garde `article_type` (ex. 'Jackets') -> type fin de l'ItemCard lu chez les voisins, cohérent avec l'objectif 6."""
from __future__ import annotations
from pathlib import Path
import numpy as np, pandas as pd
try:
    from tqdm.auto import tqdm
except ImportError:            # tqdm facultatif
    def tqdm(x, **k): return x
from .config import Config
from .imaging import load_image
from .taxonomy import ART, COLOR_MAP, GENDER_MAP, USAGE_MAP

COLS = ["id", "category", "article_type", "color", "gender", "season", "occasion", "name"]


def build_catalogue():
    import kagglehub
    root = Path(kagglehub.dataset_download("paramaggarwal/fashion-product-images-small"))
    styles = next(p for p in root.rglob("styles.csv") if p.is_file()); imgdir = next(p for p in root.rglob("images") if p.is_dir())
    raw = pd.read_csv(styles, on_bad_lines="skip")
    raw["article_type"] = raw.articleType.astype(str).str.strip()
    raw["category"] = raw.article_type.str.lower().map(ART)
    raw = raw.dropna(subset=["category"]).copy(); raw["id"] = raw.id.astype(int)
    raw = raw[raw.id.map(lambda i: (imgdir / f"{i}.jpg").exists())]
    low = lambda col: raw[col].astype(str).str.strip().str.lower()
    raw["color"] = low("baseColour").map(COLOR_MAP); raw["gender"] = low("gender").map(GENDER_MAP)
    raw["season"] = low("season").map({"summer": "summer", "winter": "winter", "spring": "spring", "fall": "fall"})
    raw["occasion"] = low("usage").map(USAGE_MAP); raw["name"] = raw.productDisplayName.astype(str)
    return raw[COLS].reset_index(drop=True), imgdir


def build_or_load(cfg: Config, embedder, force: bool = False):
    """Renvoie (kb_df, E_kb, test_df, img_src). Même échantillonnage que le notebook (random_state 0 / 1) : résultats comparables."""
    out = cfg.out; KB_META, KB_EMB, TEST_META = out / "kb_meta.csv", out / "kb_emb.npy", out / "test_meta.csv"
    cat_df, img_src = build_catalogue()
    if not force and KB_META.exists() and KB_EMB.exists() and TEST_META.exists():
        kb_df = pd.read_csv(KB_META)
        if "article_type" in kb_df.columns:
            kb_df["image_path"] = kb_df.id.map(lambda i: str(img_src / f"{int(i)}.jpg"))
            return kb_df, np.load(KB_EMB), pd.read_csv(TEST_META), img_src
        print("kb_meta.csv ancien (sans article_type) -> reconstruction de la base.")
    test_df = pd.concat([d.sample(min(cfg.n_test_per_cat, len(d)), random_state=0) for _, d in cat_df.groupby("category")]).reset_index(drop=True)
    rest = cat_df[~cat_df.id.isin(test_df.id)]
    kb_df = pd.concat([d.sample(min(cfg.kb_per_cat, len(d)), random_state=1) for _, d in rest.groupby("category")]).reset_index(drop=True)

    def embed_ids(ids, desc):
        parts = []
        for i in tqdm(range(0, len(ids), 64), desc=desc): parts.append(embedder.images([load_image(img_src / f"{j}.jpg") for j in ids[i:i + 64]]))
        return np.vstack(parts)
    E_kb, E_test = embed_ids(list(kb_df.id), "Base kNN"), embed_ids(list(test_df.id), "Test")
    keep = (E_kb @ E_test.T).max(1) < cfg.dup_sim                              # anti-fuite : quasi-doublons base/test
    print(f"anti-fuite : {int((~keep).sum())} images de base retirées")
    kb_df, E_kb = kb_df[keep].reset_index(drop=True), E_kb[keep]
    kb_df.to_csv(KB_META, index=False); np.save(KB_EMB, E_kb); test_df.to_csv(TEST_META, index=False)
    kb_df["image_path"] = kb_df.id.map(lambda i: str(img_src / f"{int(i)}.jpg"))
    return kb_df, E_kb, test_df, img_src

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/agent.py
"""Agent de décision : Qwen2.5-VL + garde-fous kNN/RAG.  Logique du notebook v1 conservée, avec 3 ajouts :
  * `mode` : full | llm_only | knn_only  (références simples exigées par l'évaluation commune) ;
  * `after_rag_policy` : "knn" (v1) ou "llm" (ablation : la route knn_after_rag n'a fait que 53,8 % sur 13 images) ;
  * le vecteur de requête et la distribution du vote kNN sont renvoyés (confiance de la fiche)."""
from __future__ import annotations
import hashlib, time
import numpy as np
from .config import Config
from .imaging import img_key
from .parsing import parse_answer
from .prompts import prompt_v1, prompt_rag
from .retrieval import Retriever, knn_vote, consensus


def accuracy_ok(a, cfg): return a["valid"] and a["confidence"] >= cfg.conf_min


def hallucination(a, kv, nb, cfg):
    """Score H dans [0,1] : le LLM affirme-t-il des choses que les faits (vocabulaire, kNN, catalogue) contredisent ?"""
    flags, H = [], 0.0
    if not a["valid"]: return 1.0, ["invalid_output"]
    if a["issues"]: flags.append("out_of_vocabulary"); H += 0.25
    if kv["conf"] >= cfg.knn_min and kv["category"] != a["category"]: flags.append("contradicts_knn"); H += 0.60
    same = [n for n in nb if n["category"] == a["category"]] or nb
    for attr in ("color", "gender"):
        v, share = consensus(same, attr)
        if v and share >= cfg.attr_min and a[attr] not in ("unknown", v): flags.append(f"contradicts_neighbors_{attr}"); H += 0.20
    return min(H, 1.0), flags


def merge_attributes(a, nb, category, cfg):
    """season/occasion -> voisins d'abord (conventions du catalogue) ; color/gender -> LLM d'abord (visibles)."""
    same = [n for n in nb if n["category"] == category] or nb; out = {}
    for attr in ("color", "gender", "season", "occasion"):
        v, share = consensus(same, attr); x = a[attr] if a else "unknown"
        if attr in ("season", "occasion"): out[attr] = v if (v and share >= 0.5) else x
        else: out[attr] = x if (x != "unknown" and not (v and share >= cfg.attr_min and v != x)) else (v or x)
    return out


class RecognitionAgent:
    def __init__(self, llm, embedder, retriever: Retriever, cfg: Config, cache=None):
        self.llm, self.embedder, self.retriever, self.cfg, self.cache = llm, embedder, retriever, cfg, cache

    def _llm(self, image, prompt):
        key = hashlib.md5(prompt.encode()).hexdigest()[:10] + "|" + img_key(image)
        if self.cache is not None and key in self.cache: raw, lat = self.cache[key]["raw"], self.cache[key]["latency"]
        else:
            t = time.time()
            try: raw = self.llm.generate(image, prompt)
            except Exception as e:
                a = parse_answer(""); a["issues"] = ["llm_error: " + repr(e)[:80]]; return a
            lat = time.time() - t
            if self.cache is not None: self.cache.put(key, raw=raw, latency=lat)
        a = parse_answer(raw); a["latency"] = lat; return a

    def run(self, image, mode: str = "full", query_emb=None) -> dict:
        cfg = self.cfg; tr = []; t0 = time.time(); calls = 0
        q = query_emb if query_emb is not None else self.embedder.images([image])[0]
        nb = self.retriever.search(q, cfg.knn_k); kv = knn_vote(nb)
        tr.append(f"OUTIL knn_search        → {kv['category']} (sûr à {kv['conf']:.0%})")

        def done(cat, conf, source, ans, status="OK", H=0.0, flags=(), llm_attrs=True):
            r = dict(category=cat, confidence=float(conf), source=source, status=status, H=float(H), flags=list(flags), llm_calls=calls,
                     latency_s=time.time() - t0, trace=tr, first=first, neighbors=nb, knn=kv, query_emb=q, llm_conf=(ans or {}).get("confidence", 0.0),
                     evidence=(ans or {}).get("evidence", ""), candidate=(ans or {}).get("category"))
            use = ans if (llm_attrs and ans and ans["valid"]) else None
            r.update(merge_attributes(use, nb, cat if cat != "unknown" else kv["category"], cfg)); return r

        if mode == "knn_only":
            first = dict(valid=False, category="invalid", color="unknown", gender="unknown", issues=[])
            tr.append("MODE knn_only"); return done(kv["category"], kv["conf"], "knn_only", None, llm_attrs=False)

        a1 = self._llm(image, prompt_v1(cfg.tunisian_prompt)); calls += 1; H1, f1 = hallucination(a1, kv, nb, cfg); acc1 = accuracy_ok(a1, cfg)
        tr.append(f"OUTIL llm_analyze (v1)  → {a1['category']} conf={a1['confidence']:.2f} | précision_ok={acc1} | H={H1:.2f} {f1}")
        first = dict(valid=a1["valid"], category=a1["category"], color=a1["color"], gender=a1["gender"], issues=a1["issues"])

        if mode == "llm_only":
            tr.append("MODE llm_only")
            if a1["valid"]:
                r = done(a1["category"], a1["confidence"], "llm_only", a1, H=H1, flags=f1)
                for k in ("color", "gender", "season", "occasion"): r[k] = a1[k]         # pas de fusion avec les voisins
                return r
            return done("unknown", 0.0, "llm_only", a1, status="REVIEW", H=H1, flags=f1)

        if acc1 and H1 < cfg.h_high:                                           # niveau 0 : le LLM est fiable
            tr.append("DÉCISION : LLM fiable → réponse directe"); return done(a1["category"], a1["confidence"], "llm", a1, H=H1, flags=f1)
        if H1 < cfg.h_high:                                                    # niveau 1 : précision douteuse, pas d'hallucination avérée
            tr.append("précision douteuse → on interroge le kNN")
            if kv["conf"] >= cfg.knn_min:
                tr.append("DÉCISION : kNN sûr → réponse du kNN"); return done(kv["category"], kv["conf"], "knn_fallback", a1, H=H1, flags=f1)
            tr.append("kNN pas assez sûr → RAG")
        else: tr.append("hallucination élevée → RAG (voisins du catalogue injectés dans le prompt)")
        a2 = self._llm(image, prompt_rag(nb[:cfg.rag_k], a1, kv, f1, cfg.tunisian_prompt)); calls += 1; H2, f2 = hallucination(a2, kv, nb, cfg); acc2 = accuracy_ok(a2, cfg)
        tr.append(f"OUTIL llm_analyze (RAG) → {a2['category']} conf={a2['confidence']:.2f} | précision_ok={acc2} | H={H2:.2f} {f2}")
        if acc2 and H2 < cfg.h_high:
            tr.append("DÉCISION : réponse LLM + RAG validée"); return done(a2["category"], a2["confidence"], "llm+rag", a2, H=H2, flags=f2)
        if cfg.after_rag_policy == "llm" and acc2:
            tr.append("DÉCISION : LLM+RAG gardé malgré H élevé (policy=llm)"); return done(a2["category"], a2["confidence"], "llm+rag_flagged", a2, H=H2, flags=f2)
        if kv["conf"] >= cfg.knn_min:                                          # niveau 3 : arbitrage final
            tr.append("RAG insuffisant, kNN sûr → réponse du kNN"); return done(kv["category"], kv["conf"], "knn_after_rag", a2 if a2["valid"] else a1, H=H2, flags=f2)
        tr.append("DÉCISION : aucun avis fiable → REVIEW (revue humaine)")
        return done("unknown", 0.0, "review", a2 if a2["valid"] else a1, status="REVIEW", H=H2, flags=f2)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/itemcard_builder.py
"""Prédictions de l'agent  ->  ItemCard v1 (+ confiances par attribut, statut, raisons).

CONFIANCES : le contrat exige confidence.slot/colour/type.  Les valeurs ci-dessous sont des HEURISTIQUES non calibrées
(combinaison de la confiance du LLM, du vote kNN et, pour la couleur, du témoin de pixels). À calibrer sur les 30 photos réelles
(eval/eval_real.py imprime la table précision-par-tranche-de-confiance).  Ne jamais les présenter comme des probabilités."""
from __future__ import annotations
from contracts import (ItemCard, Confidence, slot_of, default_type, usage_out, gender_out, season_out, warmth_formality,
                       OPEN_TOE_TYPES, WATERPROOF_TYPES, SLOT_UNCONFIRMED, TRADITIONAL_SUBTYPE_TO_SLOT)
from .retrieval import attr_share


def clamp01(x) -> float:
    """Ramène une confiance dans [0, 1] : des sommes en float32 donnent parfois 1.0000000149 et Pydantic (le=1) refuse la fiche."""
    return float(min(1.0, max(0.0, float(x))))


def fine_type(category: str, nb: list[dict]) -> tuple[str, float]:
    """Type fin = articleType majoritaire (pondéré) parmi les voisins de la même catégorie ; repli : table par défaut."""
    same = [n for n in nb if n["category"] == category and n.get("article_type")]; tot = sum(n["w"] for n in same)
    if not same: return default_type(category), 1.0
    d = {}
    for n in same: d[n["article_type"]] = d.get(n["article_type"], 0.0) + n["w"]
    t = max(d, key=d.get); return t, d[t] / tot


def traditional_subtype(nb: list[dict], min_share: float = 0.5):
    """Sous-type (jebba, farmla…) majoritaire parmi les voisins traditionnels qui en ont un ; (None, 0) sinon."""
    same = [n for n in nb if n["category"] == "traditional_garment" and n.get("subtype")]; tot = sum(n["w"] for n in same)
    if not same: return None, 0.0
    d = {}
    for n in same: d[n["subtype"]] = d.get(n["subtype"], 0.0) + n["w"]
    t = max(d, key=d.get); return (t, d[t] / tot) if d[t] / tot >= min_share else (None, d[t] / tot)


def slot_confidence(category, llm_cat, knn_dist, type_conf) -> float:
    slot = slot_of(category); mass = sum(w for c, w in knn_dist.items() if slot_of(c) == slot)
    if llm_cat and slot_of(llm_cat) == slot: return float(min(1.0, max(type_conf, mass)))   # LLM et kNN d'accord sur le slot
    return float(min(type_conf, mass)) if mass > 0 else float(type_conf * 0.5)


def colour_confidence(colour, llm_conf, nb, category, pixel_cols) -> float:
    if colour == "unknown": return 0.0
    same = [n for n in nb if n["category"] == category] or nb
    share = attr_share(same, "color", colour)
    if pixel_cols:
        names = [c for c, _ in pixel_cols]; pix = 1.0 if names[0] == colour else (0.5 if colour in names else 0.0)
        return float(0.4 * llm_conf + 0.3 * share + 0.3 * pix)
    return float(0.5 * llm_conf + 0.5 * share)


def build_card(item_id: str, res: dict, *, image: str, image_crop: str, embedding: str, pixel_cols=None,
               crop_quality: dict | None = None, review_conf: float = 0.60, second_colour: bool = False,
               subtype_min_share: float = 0.5, confirm_traditional: bool = True):
    """Renvoie (card|None, status, reasons)."""
    cat = res["category"]; reasons = []; nb = res["neighbors"]
    if cat == "unknown": cat = res["knn"]["category"]; reasons.append("agent_review_used_knn_top")
    slot = slot_of(cat)
    if slot is None: return None, "unsupported", [f"no_slot_for_category:{cat}"]
    type_, tshare = fine_type(cat, nb)
    base = float(res["confidence"]) if res["status"] == "OK" else 0.0
    type_conf = float(min(1.0, base * tshare)); slot_conf = slot_confidence(cat, res.get("candidate") or None, res["knn"]["dist"], base)
    if cat == "traditional_garment":                                         # sous-type lu chez les voisins tunisiens (si le jeu en contient)
        sub, sshare = traditional_subtype(nb, subtype_min_share)
        if sub:
            slot = TRADITIONAL_SUBTYPE_TO_SLOT.get(sub, "onepiece"); type_ = sub.title(); type_conf = float(min(1.0, base * sshare))
            slot_conf = float(min(slot_conf, sshare)); reasons.append(f"traditional_subtype_from_neighbours:{sub}")
        else: reasons.append("traditional_subtype_unknown")
    colour = res["color"] if res["color"] != "unknown" else (pixel_cols[0][0] if pixel_cols else "multicolor")
    if res["color"] == "unknown": reasons.append("colour_from_pixels_or_default")
    col_conf = colour_confidence(res["color"], float(res.get("llm_conf", 0.0)), nb, cat, pixel_cols)
    colours = [colour]
    if second_colour and pixel_cols and len(pixel_cols) > 1 and pixel_cols[1][1] >= 0.25 and pixel_cols[1][0] != colour: colours.append(pixel_cols[1][0])
    w, f, official = warmth_formality(cat, type_)
    if not official: reasons.append("warmth_formality_placeholder")
    if cat in SLOT_UNCONFIRMED: reasons.append(f"slot_mapping_unconfirmed:{cat}")
    if res["occasion"] == "home": reasons.append("usage_home_mapped_to_Casual")
    if crop_quality and not crop_quality["ok"]: reasons += [f"crop:{r}" for r in crop_quality["reasons"]]
    tl = type_.lower()
    card = ItemCard(id=item_id, name=f"{colour.title()} {cat.replace('_', ' ')}", slot=slot, type=type_, colour=colours, usage=usage_out(res["occasion"]),
                    warmth=w, formality=f, waterproof=tl in WATERPROOF_TYPES, open_toe=tl in OPEN_TOE_TYPES, gender=gender_out(res["gender"]),
                    image=image, image_crop=image_crop, embedding=embedding, season=season_out(res["season"]), pattern="plain",
                    confidence=Confidence(slot=clamp01(slot_conf), colour=clamp01(col_conf), type=clamp01(type_conf)),
                    source="obj1.knn" if "knn" in res["source"] else "obj1.llm", traditional=(cat == "traditional_garment"))
    if res["status"] != "OK": reasons.append("agent_could_not_decide")
    if min(slot_conf, type_conf) < review_conf: reasons.append("low_confidence")
    review = (res["status"] != "OK" or min(slot_conf, type_conf) < review_conf or (cat in SLOT_UNCONFIRMED and confirm_traditional)
              or bool(crop_quality and not crop_quality["ok"]))
    return card, ("needs_review" if review else "ok"), reasons


def apply_correction(card: ItemCard, *, slot=None, type_=None, colour=None, usage=None) -> ItemCard:
    """Correction utilisateur : source='user_corrected', confiance 1.0 sur les champs corrigés (contrat v1)."""
    d = card.model_dump(); conf = d["confidence"]
    if slot: d["slot"] = slot; conf["slot"] = 1.0
    if type_: d["type"] = type_; conf["type"] = 1.0
    if colour: d["colour"] = [colour] if isinstance(colour, str) else list(colour); conf["colour"] = 1.0
    if usage: d["usage"] = usage
    d["source"] = "user_corrected"; return ItemCard(**d)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/outfit.py
"""Cohérence entre la SEGMENTATION (ce que le crop croit avoir découpé) et la RECONNAISSANCE (ce que l'agent voit dans le crop).

Pourquoi ce module : SegFormer est un modèle de « parsing » de vêtements occidentaux (classes : upper-clothes, pants, skirt, dress, shoes, bag, hat…).
Il n'a pas de classe « veste » (les vestes sont dans upper-clothes) ni de classe pour une jebba ou une farmla : une longue pièce traditionnelle est
souvent coupée en plusieurs morceaux (haut + bas + « chaussures » + « écharpe »…), et chaque morceau, vu seul, est reconnu comme la MÊME pièce longue.

Règle : on compare la classe de segmentation (`hint`) au slot reconnu.
  * compatible  -> rien à signaler (ex. « upper-clothes » reconnu comme veste ou comme robe : normal) ;
  * PETITE pièce (chaussures, sac, accessoire) reconnue comme tout autre chose -> c'est un FRAGMENT d'un autre vêtement : on l'ignore ;
  * GRANDE pièce (haut, bas, robe) incompatible -> on garde la fiche mais elle est marquée `slot_disagrees_with_segmentation` (revue).
La table COMPATIBLE est un choix de conception (non validé statistiquement) : à ajuster d'après les résultats."""
from __future__ import annotations

COMPATIBLE = {
    "top": {"top", "outer", "onepiece"},          # upper-clothes contient aussi vestes/gilets ; une robe ou une tunique longue commence au haut du corps
    "bottom": {"bottom", "onepiece"},             # le bas d'une robe / jebba est « skirt » ou « pants » pour SegFormer
    "onepiece": {"onepiece", "top", "outer"},
    "footwear": {"footwear"},
    "bag": {"bag", "accessory"},
    "accessory": {"accessory", "bag"},
}
SMALL_HINTS = {"footwear", "bag", "accessory"}


def judge_piece(hint: str | None, slot: str | None) -> str:
    """'ok' | 'inconsistent' (grande pièce, à revoir) | 'fragment' (petite pièce reconnue comme autre chose : à ignorer)."""
    if not hint or not slot or slot in COMPATIBLE.get(hint, {slot}): return "ok"
    return "fragment" if hint in SMALL_HINTS else "inconsistent"

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/tunisian.py
"""Jeu de données tunisien : à REMPLIR par toi, utilisé pour (A) références kNN/RAG sans entraînement, (C) fine-tuning, (D) évaluation.

Dossier :   <DRESSME_ROOT>/tunisian_dataset/
                images/…            tes photos (n'importe quel sous-dossier)
                labels.csv          une ligne par photo (créé / complété par `make_template`)
                crops/              crops 512 px générés automatiquement (ne pas toucher)

labels.csv — colonnes (seules `image_path` et `category` sont obligatoires ; une ligne sans `category` est ignorée = « à faire ») :
    image_path   chemin relatif à tunisian_dataset/ (ex. images/jebba_01.jpg)
    category     UNE des 17 classes (tshirt, shirt, … , traditional_garment)        ← la vérité terrain
    subtype      facultatif : jebba | farmla | blouza | kaftan | fouta | other      (surtout pour traditional_garment)
    colour       facultatif : beige black blue brown green grey metallic multicolor orange pink purple red white yellow
    gender, season, occasion   facultatifs (mêmes valeurs que le notebook)
    event, origin              facultatifs (libre : wedding, tunisia…)
    group_id     IMPORTANT : même valeur pour les photos d'une même séance / tenue / personne (évite qu'elles soient des deux côtés d'un split)
    split        facultatif : train | dev | test.  Vide -> attribution automatique PAR GROUPE (60 / 20 / 20, déterministe)
    cropped      facultatif : 1 si la photo est déjà un crop (sinon le crop automatique est appliqué)

Règles anti-biais : `train` = références + fine-tuning ; `dev` = réglage des seuils ; `test` = regardé UNE fois, à la fin.
Les 30 photos de l'évaluation finale ne doivent PAS être dans ce fichier avec un split `train`."""
from __future__ import annotations
import hashlib
from pathlib import Path
import numpy as np, pandas as pd
from contracts import slot_of, TRADITIONAL_SUBTYPE_TO_SLOT
from .config import Config
from PIL import Image
from .imaging import normalise_crop, open_rgb
from .taxonomy import CATEGORIES, COLORS, GENDERS, SEASONS, OCCASIONS, TN_SUBTYPES

COLS = ["image_path", "category", "subtype", "colour", "gender", "season", "occasion", "event", "origin", "group_id", "split", "cropped"]
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".heic", ".heif"}
_READ_CACHE: dict = {}


def readable(path) -> tuple[bool, str]:
    """True si l'image est RÉELLEMENT décodable (pas seulement identifiable : `verify()` laisse passer des fichiers tronqués).
    Cas typiques d'échec sur Drive : fichier de 0 octet, téléversement interrompu, HEIC renommé en .jpg (installe pillow-heif)."""
    p = Path(path)
    try: st = p.stat(); key = (str(p), st.st_mtime, st.st_size)
    except OSError as e: return False, f"inaccessible ({e.__class__.__name__})"
    if key in _READ_CACHE: return _READ_CACHE[key]
    if st.st_size == 0: res = (False, "fichier vide (0 octet)")
    else:
        try:
            with Image.open(p) as im: im.load()
            res = (True, "")
        except Exception as e: res = (False, f"{e.__class__.__name__}: {str(e)[:80]}")
    _READ_CACHE[key] = res; return res


def tn_dir(cfg: Config) -> Path:
    d = cfg.root / "tunisian_dataset"; (d / "images").mkdir(parents=True, exist_ok=True); return d


def make_template(cfg: Config) -> dict:
    """Crée labels.csv, ou AJOUTE une ligne vide pour chaque nouvelle image trouvée (n'écrase jamais ce que tu as rempli)."""
    d = tn_dir(cfg); csv = d / "labels.csv"
    old = pd.read_csv(csv, dtype=str).fillna("") if csv.exists() else pd.DataFrame(columns=COLS)
    for c in COLS:
        if c not in old: old[c] = ""
    have = set(old.image_path.map(lambda p: Path(p).as_posix()))
    new = sorted(p.relative_to(d).as_posix() for p in d.rglob("*") if p.suffix.lower() in IMG_EXT and "crops" not in p.parts and p.relative_to(d).as_posix() not in have)
    add = pd.DataFrame([{**{c: "" for c in COLS}, "image_path": p, "origin": "tunisia"} for p in new])
    pd.concat([old[COLS], add], ignore_index=True).to_csv(csv, index=False)
    return dict(images_total=len(have) + len(new), nouvelles_lignes=len(new), fichier=str(csv))


def _group_split(group: str) -> str:
    h = int(hashlib.md5(group.encode()).hexdigest(), 16) % 100
    return "train" if h < 60 else "dev" if h < 80 else "test"


def load_tunisian(cfg: Config, split: str | None = None):
    """Renvoie (df, issues). df ne contient que des lignes VALIDES (fichier présent, catégorie connue) avec `split` renseigné.
    issues = liste de (niveau, message) ; niveau 'ERREUR' = ligne ignorée, 'ATTENTION' = à corriger."""
    d = tn_dir(cfg); csv = d / "labels.csv"; issues = []
    if not csv.exists(): return pd.DataFrame(columns=COLS + ["abs_path"]), [("ATTENTION", f"{csv} absent : lance make_template(cfg)")]
    df = pd.read_csv(csv, dtype=str).fillna("")
    for c in COLS:
        if c not in df: df[c] = ""
    n_pending = int((df.category.str.strip() == "").sum()); df = df[df.category.str.strip() != ""].copy()
    if n_pending: issues.append(("INFO", f"{n_pending} lignes sans category : ignorées (à remplir)"))
    if df.empty: return pd.DataFrame(columns=COLS + ["abs_path"]), issues                     # rien d'étiqueté pour l'instant
    for c in ("category", "subtype", "colour", "gender", "season", "occasion", "split"): df[c] = df[c].str.strip().str.lower()
    df["abs_path"] = df.image_path.map(lambda p: str(Path(p) if Path(p).is_absolute() else d / p))
    bad = ~df.abs_path.map(lambda p: Path(p).is_file())
    for p in df.image_path[bad]: issues.append(("ERREUR", f"fichier introuvable : {p}"))
    df = df[~bad]
    if df.empty: return pd.DataFrame(columns=COLS + ["abs_path"]), issues
    ok = [readable(p) for p in df.abs_path]                                                    # décodage réel de chaque image
    for p, (good, why) in zip(df.image_path, ok):
        if not good: issues.append(("ERREUR", f"image illisible : {p} — {why}"))
    df = df[[g for g, _ in ok]]
    if df.empty: return pd.DataFrame(columns=COLS + ["abs_path"]), issues
    bad = ~df.category.isin(CATEGORIES)
    for p, c in zip(df.image_path[bad], df.category[bad]): issues.append(("ERREUR", f"category invalide '{c}' ({p}) : doit être l'une des 17 classes"))
    df = df[~bad]
    for col, allowed in (("subtype", TN_SUBTYPES), ("colour", COLORS), ("gender", GENDERS), ("season", SEASONS), ("occasion", ["casual", "formal", "sport", "party", "traditional", "home"])):
        b = (df[col] != "") & ~df[col].isin(allowed)
        for p, v in zip(df.image_path[b], df[col][b]): issues.append(("ATTENTION", f"{col} invalide '{v}' ({p}) : ignoré"))
        df.loc[b, col] = ""
    dup = df.image_path.duplicated(keep="first")
    for p in df.image_path[dup]: issues.append(("ERREUR", f"image listée deux fois : {p}"))
    df = df[~dup].copy()
    df["group_id"] = [g if g.strip() else Path(p).stem for g, p in zip(df.group_id, df.image_path)]
    if (df.group_id == df.image_path.map(lambda p: Path(p).stem)).all() and len(df) > 3:
        issues.append(("ATTENTION", "aucun group_id renseigné : des photos d'une même séance peuvent se retrouver des deux côtés d'un split"))
    auto = ~df.split.isin(["train", "dev", "test"]); df.loc[auto, "split"] = df.group_id[auto].map(_group_split)
    df["md5"] = df.abs_path.map(lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest())            # fichiers identiques -> même split (anti-fuite)
    first = df.groupby("md5").split.transform("first")
    if (first != df.split).any(): issues.append(("ATTENTION", f"{int((first != df.split).sum())} doublons exacts répartis sur plusieurs splits : remis dans le même split"))
    df["split"] = first
    n = df.groupby(["category", "split"]).size().unstack(fill_value=0)
    for c in n.index:
        if n.loc[c].sum() < 3: issues.append(("ATTENTION", f"classe '{c}' : seulement {int(n.loc[c].sum())} photo(s) — trop peu pour une conclusion"))
    out = df.drop(columns="md5").reset_index(drop=True)
    return (out[out.split == split].reset_index(drop=True) if split else out), issues


def report(cfg: Config) -> pd.DataFrame:
    df, issues = load_tunisian(cfg)
    for lvl, msg in issues: print(f"[{lvl}] {msg}")
    if len(df): tab = df.groupby(["category", "split"]).size().unstack(fill_value=0); print(tab.to_string()); print("total :", len(df)); return tab
    print("aucune ligne valide pour l'instant."); return pd.DataFrame()


def _target_slot(r) -> str | None:
    return TRADITIONAL_SUBTYPE_TO_SLOT.get(r.subtype) if r.category == "traditional_garment" and r.subtype else slot_of(r.category)


def prepare_crops(cfg: Config, df: pd.DataFrame, cropper=None, force: bool = False) -> pd.DataFrame:
    """Un crop 512 px par photo (mis en cache). Les références et la requête passent ainsi par le MÊME prétraitement.
    Une photo qui échoue (illisible, crop impossible) est IGNORÉE avec un avertissement : elle ne fait plus planter le chargement."""
    out = tn_dir(cfg) / "crops"; out.mkdir(exist_ok=True); paths, failed = [], []
    for r in df.itertuples():
        tgt = out / (hashlib.md5(r.abs_path.encode()).hexdigest()[:12] + "_crop.png")
        try:
            if force or not tgt.exists():
                im = open_rgb(r.abs_path)
                if str(r.cropped).strip().lower() in ("1", "true", "yes", "oui") or cropper is None: crop = normalise_crop(im, cfg.crop_size)
                else:
                    pieces = cropper.crop(im, cfg.crop_mode); want = _target_slot(r)
                    match = [p for p in pieces if p.slot_hint == want] or pieces
                    crop = max(match, key=lambda p: p.area_frac).image
                crop.save(tgt)
            paths.append(str(tgt))
        except Exception as e:
            paths.append(None); failed.append((r.image_path, f"{e.__class__.__name__}: {str(e)[:80]}"))
    for p, why in failed: print(f"⚠️ photo ignorée (crop impossible) : {p} — {why}")
    df = df.copy(); df["crop_path"] = paths; df = df[df.crop_path.notna()].reset_index(drop=True); df.attrs["failed"] = failed; return df


def add_tunisian_references(cfg: Config, embedder, kb_df: pd.DataFrame, E_kb: np.ndarray, cropper=None, tag: str = "frozen"):
    """Ajoute les photos tunisiennes `train` comme voisins du kNN/RAG (SANS entraînement).  Renvoie (kb_df, E, n_ajoutées).
    `embedder` = l'encodeur qui a servi à construire E_kb (gelé, ou affiné s'il y a fine-tuning) ; `tag` identifie le cache."""
    df, _ = load_tunisian(cfg, "train")
    if not len(df): return kb_df, E_kb, 0
    df = prepare_crops(cfg, df, cropper)
    key = hashlib.md5((tag + "|".join(f"{p}:{Path(p).stat().st_mtime}:{c}" for p, c in zip(df.crop_path, df.category))).encode()).hexdigest()[:10]
    cache = cfg.out / f"tn_refs_{key}.npy"
    if cache.exists(): Et = np.load(cache)
    else:
        Et = np.vstack([embedder.images([open_rgb(p) for p in df.crop_path.iloc[i:i + 32]], bs=16) for i in range(0, len(df), 32)]); np.save(cache, Et)
    t = pd.DataFrame(dict(id=-1 - np.arange(len(df)), category=df.category, article_type=None, subtype=df.subtype.replace("", None), color=df.colour.replace("", None),
                          gender=df.gender.replace("", None), season=df.season.replace("", None), occasion=df.occasion.replace("", None),
                          name=[f"Tunisian {s or c}" for s, c in zip(df.subtype, df.category)], image_path=df.crop_path))
    return pd.concat([kb_df, t], ignore_index=True), np.vstack([E_kb, Et]), len(df)

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/finetune.py
"""Fine-tuning OPTIONNEL de FashionCLIP (dernier bloc vision + projection + tête de classes), dataset mixte catalogue + photos tunisiennes.

Différences voulues par rapport au notebook v1 :
  * le modèle affiné est une COPIE (`embedder.clone()`) : l'encodeur gelé qui produit les embeddings des fiches n'est jamais modifié
    (sinon ses vecteurs ne seraient plus comparables à ceux des objectifs 2-6) ;
  * GradScaler pour l'entraînement fp16 (le notebook faisait un autocast fp16 sans mise à l'échelle du gradient) ;
  * rien n'est lancé sans `tunisian_dataset/labels.csv` (aucune donnée inventée).
Données tunisiennes : voir recognition/tunisian.py.  Le fine-tuning n'utilise QUE le split `train` (crops 512 px) ;
`dev` règle les seuils, `test` n'est jamais vu.  Les références kNN tunisiennes sont ajoutées à part (tunisian.add_tunisian_references)."""
from __future__ import annotations
import hashlib, random
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from .config import Config
from .imaging import load_image, open_rgb
from .taxonomy import CATEGORIES
from .tunisian import load_tunisian, prepare_crops

EPOCHS, BATCH, LR, HEAD_LR, PATIENCE, VAL_FRAC, TN_REPEAT, MAX_GENERIC = 5, 8, 1e-5, 2e-4, 2, 0.2, 2, 250


def _paths(cfg: Config):
    d = cfg.out / "fashionclip_finetuned"; d.mkdir(exist_ok=True); return d, d / "fashionclip_visual_finetuned.pt"


def build_mixed_dataset(cfg: Config, kb_df: pd.DataFrame, cropper=None):
    g = kb_df[kb_df.id >= 0].copy(); g["origin"] = "catalogue"; g["group_id"] = g.id.map(lambda x: f"catalogue_{int(x)}"); g["event"] = ""
    g = pd.concat([x.sample(min(len(x), MAX_GENERIC), random_state=cfg.seed) for _, x in g.groupby("category")], ignore_index=True)
    tn, issues = load_tunisian(cfg, "train")
    for lvl, msg in issues:
        if lvl == "ERREUR": print(f"[{lvl}] {msg}")
    if not len(tn): return g, pd.DataFrame(columns=g.columns)
    tn = prepare_crops(cfg, tn, cropper); tn = tn.assign(image_path=tn.crop_path, origin="tunisia")
    return g, tn[[c for c in tn.columns if c in set(g.columns) | {"subtype", "colour"}]].reset_index(drop=True)


def split_grouped(df: pd.DataFrame, seed: int):
    from sklearn.model_selection import GroupShuffleSplit
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=seed).split(df, groups=df.group_id.astype(str)))
    return df.iloc[a].copy(), df.iloc[b].copy()


def train(cfg: Config, base_embedder, kb_df: pd.DataFrame, cropper=None):
    import torch
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import f1_score, accuracy_score
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    generic, tn = build_mixed_dataset(cfg, kb_df, cropper)
    if len(tn) == 0: print("Aucune photo tunisienne `train` valide : aucun fine-tuning (rien n'est inventé)."); return None
    combined = pd.concat([generic, tn], ignore_index=True).drop_duplicates("image_path").reset_index(drop=True)
    tr, va = split_grouped(combined, cfg.seed)
    tr = pd.concat([tr] + [tr[tr.origin == "tunisia"]] * (TN_REPEAT - 1), ignore_index=True)      # sur-échantillonnage TRAIN uniquement
    print("train", len(tr), "val", len(va), "| tunisien train", int((tr.origin == "tunisia").sum()), "val", int((va.origin == "tunisia").sum()))
    emb = base_embedder.clone(); model = emb.model; dev = emb.dev; d, weights = _paths(cfg)

    class DS(Dataset):
        def __init__(s, df): s.df = df.reset_index(drop=True)
        def __len__(s): return len(s.df)
        def __getitem__(s, i): r = s.df.iloc[i]; return open_rgb(r.image_path), CATEGORIES.index(r.category)
    def collate(b):
        im, y = zip(*b); return emb.proc(images=list(im), return_tensors="pt")["pixel_values"], torch.tensor(y)
    def feats(px):
        f = model.get_image_features(pixel_values=px); f = f.pooler_output if hasattr(f, "pooler_output") else f
        return torch.nn.functional.normalize(f.float(), dim=-1)

    dl_tr = DataLoader(DS(tr), batch_size=BATCH, shuffle=True, collate_fn=collate); dl_va = DataLoader(DS(va), batch_size=BATCH, collate_fn=collate)
    for p in model.parameters(): p.requires_grad = False
    trainables = list(model.vision_model.encoder.layers[-1].parameters()) + list(model.vision_model.post_layernorm.parameters()) + list(model.visual_projection.parameters())
    for p in trainables: p.requires_grad = True
    head = torch.nn.Linear(model.config.projection_dim, len(CATEGORIES)).to(dev); cnt = Counter(tr.category)
    w = torch.tensor([min(5.0, max(0.35, len(tr) / (len(CATEGORIES) * max(1, cnt.get(c, 0))))) for c in CATEGORIES], device=dev)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW([{"params": trainables, "lr": LR}, {"params": head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda"); best, wait, rows = -1.0, 0, []
    for ep in range(1, EPOCHS + 1):
        model.train(); head.train(); tot = seen = 0
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"): loss = loss_fn(head(feats(x)), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables + list(head.parameters()), 1.0)
            scaler.step(opt); scaler.update(); tot += loss.item() * len(y); seen += len(y)
        model.eval(); head.eval(); T, P = [], []
        with torch.no_grad():
            for x, y in dl_va: P += head(feats(x.to(dev))).argmax(-1).cpu().tolist(); T += y.tolist()
        mf = f1_score(T, P, average="macro", zero_division=0); rows.append(dict(epoch=ep, train_loss=tot / max(1, seen), val_acc=accuracy_score(T, P), val_macro_f1=mf)); print(rows[-1])
        if mf > best + 1e-5:
            best, wait = mf, 0
            torch.save({"vision_last": model.vision_model.encoder.layers[-1].state_dict(), "post_norm": model.vision_model.post_layernorm.state_dict(),
                        "projection": model.visual_projection.state_dict(), "head": head.state_dict(), "categories": CATEGORIES}, weights)
        else:
            wait += 1
            if wait >= PATIENCE: print("early stopping"); break
    pd.DataFrame(rows).to_csv(d / "training_history.csv", index=False); return weights


def load_finetuned_retrieval(cfg: Config, shared_embedder, kb_df: pd.DataFrame, img_src, cropper=None):
    """Charge l'encodeur affiné (COPIE) et ré-indexe la base catalogue.  Renvoie (encodeur_affiné, kb_df, E_kb_affiné, tag).
    Les fiches continuent d'utiliser `shared_embedder` (gelé).  Si aucun entraînement n'est possible : (encodeur gelé, kb_df, E gelé, 'frozen')."""
    import torch
    d, weights = _paths(cfg)
    if not weights.exists():
        w = train(cfg, shared_embedder, kb_df, cropper)
        if w is None: return shared_embedder, kb_df, np.load(cfg.out / "kb_emb.npy"), "frozen"
    ck = torch.load(weights, map_location="cpu", weights_only=False)
    if ck["categories"] != CATEGORIES: raise ValueError("taxonomie du checkpoint différente")
    emb = shared_embedder.clone(); m = emb.model
    m.vision_model.encoder.layers[-1].load_state_dict(ck["vision_last"]); m.vision_model.post_layernorm.load_state_dict(ck["post_norm"])
    m.visual_projection.load_state_dict(ck["projection"]); m.eval()
    tag = hashlib.sha256(weights.read_bytes()).hexdigest()[:12]; path = d / f"kb_finetuned_{tag}.npy"
    base = kb_df[kb_df.id >= 0]
    if path.exists(): E = np.load(path)
    else: E = np.vstack([emb.images([load_image(p) for p in base.image_path.iloc[j:j + 32]], bs=16) for j in range(0, len(base), 32)]); np.save(path, E)
    return emb, base.reset_index(drop=True), E, "ft_" + tag

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/service.py
"""Service de reconnaissance (Objectif 1) : crop partagé  ->  ItemCard v1 + embedding .npy.

    svc = RecognitionService.from_config(CFG)                       # charge Qwen + FashionCLIP + base kNN (GPU)
    r = svc.recognize("crops/veste.png", item_id="it_0042", out_dir="/chemin/store")
    r.status, r.card, r.reasons
"""
from __future__ import annotations
import hashlib, json
from pathlib import Path
import numpy as np
from contracts import RecognitionResult
from .colour import dominant_colours
from .crop import Cropper
from .config import Config, CFG
from .imaging import load_image, normalise_crop, open_rgb, check_crop
from .itemcard_builder import build_card
from .outfit import judge_piece


class RecognitionService:
    def __init__(self, agent, shared_embedder, cfg: Config = CFG, shared_hook=None, mode: str = "full", cropper=None):
        self.agent, self.shared, self.cfg, self.hook, self.mode, self.cropper = agent, shared_embedder, cfg, shared_hook, mode, cropper

    # ---------- chargement complet (GPU) ----------
    @classmethod
    def from_config(cls, cfg: Config = CFG, mode: str = "full", use_finetuned: bool = False, load_llm: bool = True):
        from .agent import RecognitionAgent
        from .embedder import FashionClipEmbedder, load_shared_embed_hook
        from .knowledge_base import build_or_load
        from .llm import VisionLLM, JsonlCache
        from .retrieval import Retriever
        from .tunisian import add_tunisian_references
        shared = FashionClipEmbedder(cfg.embedder_id)                       # GELÉ : sert aux fiches ET (par défaut) au kNN
        cropper = Cropper.from_config(cfg)
        kb_df, E_kb, test_df, img_src = build_or_load(cfg, shared)
        retr_emb, tag = shared, "frozen"
        if use_finetuned:                                                    # l'encodeur affiné ne sert qu'au kNN interne
            from .finetune import load_finetuned_retrieval
            retr_emb, kb_df, E_kb, tag = load_finetuned_retrieval(cfg, shared, kb_df, img_src, cropper)
        retr_base = Retriever(E_kb, kb_df); retr_ref, n_tn = retr_base, 0
        if cfg.tn_refs:
            kb2, E2, n_tn = add_tunisian_references(cfg, retr_emb, kb_df, E_kb, cropper, tag)
            if n_tn: retr_ref = Retriever(E2, kb2)
        print(f"références kNN : {len(kb_df)} catalogue + {n_tn} tunisiennes (train)")
        # load_llm=False : PC sans GPU NVIDIA -> pas de Qwen ; kNN FashionCLIP seul (mode « knn_only », moins précis : voir README)
        llm = VisionLLM(cfg.llm_id, cfg.llm_4bit, cfg.max_new_tokens) if load_llm else None
        if not load_llm: mode = "knn_only"
        agent = RecognitionAgent(llm, retr_emb, retr_ref, cfg, JsonlCache(cfg.out / "llm_calls.jsonl"))
        svc = cls(agent, shared, cfg, load_shared_embed_hook(), mode)
        svc.test_df, svc.img_src, svc.kb_df, svc.cropper, svc.retr_base, svc.retr_ref, svc.n_tn_refs = test_df, img_src, kb_df, cropper, retr_base, retr_ref, n_tn
        return svc

    def enable_finetuned(self) -> bool:
        """Active l'encodeur FashionCLIP affiné (entraîné sur les photos tunisiennes `train`) POUR LE kNN INTERNE, sans recharger Qwen.
        Les fiches gardent l'embedding du FashionCLIP gelé.  Renvoie False si aucun entraînement n'a été possible (pas de photos train)."""
        from .finetune import load_finetuned_retrieval
        from .retrieval import Retriever
        from .tunisian import add_tunisian_references
        emb, kb, E, tag = load_finetuned_retrieval(self.cfg, self.shared, self.kb_df, self.img_src, self.cropper)
        if tag == "frozen": print("fine-tuning impossible : pas assez de photos tunisiennes `train` valides."); return False
        base = Retriever(E, kb); ref = base
        if self.cfg.tn_refs:
            kb2, E2, n = add_tunisian_references(self.cfg, emb, kb, E, self.cropper, tag)
            if n: ref = Retriever(E2, kb2)
        self.agent.embedder, self.agent.retriever, self.retr_base, self.retr_ref = emb, ref, base, ref
        print("encodeur affiné actif pour le kNN (", tag, ")"); return True

    # ---------- embedding commun ----------
    def embed_crop(self, crop_img) -> np.ndarray:
        v = np.asarray(self.hook(crop_img) if self.hook else self.shared.images([crop_img])[0], dtype="float32").reshape(-1)
        assert v.shape == (512,), f"embedding de forme {v.shape}, attendu (512,)"
        return v / (np.linalg.norm(v) + 1e-12)

    # ---------- reconnaissance ----------
    def recognize(self, crop_path, item_id: str | None = None, raw_path: str | None = None, out_dir: str | Path | None = None,
                  mode: str | None = None, crop_reasons: list | None = None, slot_hint: str | None = None) -> RecognitionResult:
        cfg = self.cfg; crop_path = Path(crop_path)
        item_id = item_id or "it_" + hashlib.md5(str(crop_path).encode()).hexdigest()[:8]
        quality = check_crop(crop_path); crop512 = normalise_crop(open_rgb(crop_path), cfg.crop_size)
        pixel = dominant_colours(crop512)
        shared_vec = self.embed_crop(crop512)
        same_enc = getattr(self.agent, "embedder", None) is self.shared and self.hook is None   # sinon : encodeurs différents
        res = self.agent.run(load_image(crop512, cfg.min_side), mode or self.mode, query_emb=shared_vec if same_enc else None)
        # chemins relatifs du contrat (items/… et emb/…), fichiers écrits si out_dir est fourni
        rel_crop, rel_emb = f"items/{item_id}_crop.png", f"emb/{item_id}.npy"; rel_raw = f"items/{item_id}_raw.jpg" if raw_path else rel_crop
        if out_dir:
            out_dir = Path(out_dir); (out_dir / "items").mkdir(parents=True, exist_ok=True); (out_dir / "emb").mkdir(parents=True, exist_ok=True)
            crop512.save(out_dir / rel_crop); np.save(out_dir / rel_emb, shared_vec)
            if raw_path: open_rgb(raw_path).save(out_dir / rel_raw, quality=95)
        card, status, reasons = build_card(item_id, res, image=rel_raw, image_crop=rel_crop, embedding=rel_emb, pixel_cols=pixel,
                                           crop_quality=quality, review_conf=cfg.review_conf, second_colour=cfg.pixel_second_colour, subtype_min_share=cfg.subtype_min_share, confirm_traditional=cfg.confirm_traditional)
        reasons = reasons + [f"crop:{r}" for r in (crop_reasons or []) if f"crop:{r}" not in reasons]
        fragment = False
        verdict = judge_piece(slot_hint, card.slot if card is not None else None)
        if verdict == "inconsistent":                                        # grande pièce dont le slot contredit la segmentation : à revoir
            reasons.append(f"slot_disagrees_with_segmentation:{slot_hint}->{card.slot}"); status = "needs_review" if status == "ok" else status
        elif verdict == "fragment":                                          # petite pièce reconnue comme autre chose : fragment d'un autre vêtement
            reasons.append(f"fragment_of_other_garment:{slot_hint}->{card.slot}"); fragment = True
        if crop_reasons and status == "ok": status = "needs_review"
        if card is not None and out_dir: (out_dir / "items" / f"{item_id}.json").write_text(card.model_dump_json(indent=2), encoding="utf-8")
        return RecognitionResult(status=status, card=card, category=res["category"], reasons=reasons, trace=res["trace"],
                                 debug=dict(source=res["source"], H=res["H"], flags=res["flags"], llm_calls=res["llm_calls"], latency_s=res["latency_s"],
                                            pixel_colours=pixel, crop_quality=quality, knn_dist=res["knn"]["dist"], colour_raw=res["color"],
                                            confidence_calibrated=False, fragment=fragment))

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/batch.py
"""Traitement par lots : un dossier de PHOTOS (ou de crops) -> fiches JSON + .npy + fichiers récapitulatifs, dans le dossier `store`.

    store/
      items/it_0001.json        une fiche ItemCard v1 par article (le « résultat JSON » de chaque photo)
      items/it_0001_crop.png    crop 512 px fond blanc      items/it_0001_raw.jpg   photo d'origine
      emb/it_0001.npy           embedding FashionCLIP 512-d, norme 1
      wardrobe.json             LISTE de toutes les fiches (contrat pur) -> à donner aux objectifs 2-6
      recognition_report.json   une ligne par pièce : statut, raisons, source, H ; les pièces IGNORÉES y figurent (status « discarded »)

Les identifiants sont stables : relancer sur la même image réutilise son id ; une nouvelle image reçoit le numéro suivant.
`fresh=True` repart d'un store vide (ids it_0001…). Pièces ignorées (jamais dans wardrobe.json) :
  * fragments : petite pièce (chaussures, sac, accessoire) que la reconnaissance identifie comme un autre vêtement (voir outfit.py) ;
  * doublons : plusieurs pièces d'une même photo reconnues comme une pièce unique (robe, jebba…) -> on garde la plus grande."""
from __future__ import annotations
import json, re, shutil, time
from collections import Counter
from pathlib import Path

EXT = {".png", ".jpg", ".jpeg", ".webp", ".heic", ".heif"}


def _next_id(store: Path, used: set[str]) -> str:
    nums = [int(m.group(1)) for p in (store / "items").glob("it_*.json") if (m := re.fullmatch(r"it_(\d+)", p.stem))]
    nums += [int(m.group(1)) for i in used if (m := re.fullmatch(r"it_(\d+)", i))]
    return f"it_{(max(nums) + 1 if nums else 1):04d}"


def reset_store(store: str | Path, archive: bool = False) -> str:
    """Vide le store (items/, emb/, _tmp/, wardrobe.json, recognition_report.json). archive=True : déplace l'ancien contenu
    dans `<store>_archive_<date>` au lieu de le supprimer. Les autres dossiers (agent_final, tunisian_dataset…) ne sont JAMAIS touchés."""
    store = Path(store); store.mkdir(parents=True, exist_ok=True); names = ["items", "emb", "_tmp", "wardrobe.json", "recognition_report.json"]
    present = [n for n in names if (store / n).exists()]; msg = "store déjà propre"
    if present:
        if archive:
            dst = store.parent / f"{store.name}_archive_{time.strftime('%Y%m%d_%H%M%S')}"; dst.mkdir(parents=True)
            for n in present: shutil.move(str(store / n), str(dst / n))
            msg = f"ancien contenu archivé dans {dst}"
        else:
            for n in present:
                p = store / n; shutil.rmtree(p) if p.is_dir() else p.unlink()
            msg = f"{len(present)} élément(s) supprimé(s) : {present}"
    (store / "items").mkdir(parents=True, exist_ok=True); (store / "emb").mkdir(parents=True, exist_ok=True); return msg


def _discard(store: Path, item_id: str) -> None:
    for rel in (f"items/{item_id}.json", f"items/{item_id}_crop.png", f"items/{item_id}_raw.jpg", f"emb/{item_id}.npy"):
        (store / rel).unlink(missing_ok=True)


def recognize_folder(svc, folder: str | Path, store: str | Path, mode: str | None = None, verbose: bool = True,
                     crop: bool | str = False, fresh: bool = False, limit: int | None = None, archive: bool = False) -> dict:
    """crop=False : chaque image est DÉJÀ un crop (livré par le module crop).
       crop=True / "auto" / "worn" / "flat" : chaque image est une PHOTO ; svc.cropper la découpe (réglages : CFG.crop_*).
       fresh=True : repart d'un store propre.   limit=N : ne traite que les N premières images (démo)."""
    folder, store = Path(folder), Path(store)
    if fresh: print("🧹", reset_store(store, archive))
    store.mkdir(parents=True, exist_ok=True); (store / "items").mkdir(exist_ok=True)
    rep_path = store / "recognition_report.json"
    report = {e["source_key"]: e for e in json.loads(rep_path.read_text(encoding="utf-8"))["items"]} if rep_path.exists() else {}
    images = sorted(p for p in folder.iterdir() if p.suffix.lower() in EXT and not p.name.startswith("."))
    if not images: raise FileNotFoundError(f"Aucune image (png/jpg/jpeg/webp) dans {folder}")
    if limit: images = images[:limit]
    tmp = store / "_tmp"; tmp.mkdir(exist_ok=True); crop_mode = None if crop is False else ("auto" if crop is True else crop)
    if crop_mode is not None and getattr(svc, "cropper", None) is not None and hasattr(svc.cropper, "apply_config"): svc.cropper.apply_config(svc.cfg)
    for p in images:
        pieces = [(None, None, None, None, None)] if crop_mode is None else [(c.name, c.slot_hint, c.image, c.quality, c.area_frac) for c in svc.cropper.crop(p, crop_mode)]
        onepieces = []                                                       # (clé, aire) des pièces « robe / pièce unique » de CETTE photo
        for name, hint, img, q, area in pieces:
            key = p.name if name is None else f"{p.name}#{name}"; used = {e["id"] for e in report.values()}
            item_id = report[key]["id"] if key in report else _next_id(store, used); src = p
            if img is not None: src = tmp / f"{item_id}_piece.png"; img.save(src)
            r = svc.recognize(src, item_id=item_id, raw_path=str(p), out_dir=store, mode=mode, crop_reasons=(q or {}).get("reasons"), slot_hint=hint)
            base = dict(source_key=key, source_file=p.name, crop_piece=name, id=item_id, category=r.category, hint=hint, area=None if area is None else round(float(area), 4))
            if r.debug.get("fragment"):                                      # petite pièce reconnue comme autre chose : fragment -> ignorée
                _discard(store, item_id); report[key] = dict(base, status="discarded", reasons=r.reasons, json=None, discard_reason="fragment")
                if verbose: print(f"{key:34s} → ignorée (fragment) | {r.category:20s} | {r.reasons[-1]}")
                continue
            report[key] = dict(base, status=r.status, reasons=r.reasons, json=f"items/{item_id}.json" if r.card else None, source=r.debug.get("source"),
                               H=r.debug.get("H"), confidence_calibrated=r.debug.get("confidence_calibrated", False))
            if r.card is not None and r.card.slot == "onepiece" and area is not None: onepieces.append((key, area))
            if verbose: print(f"{key:34s} → {item_id} | {r.status:12s} | {r.category:20s} | {r.reasons}")
        if len(onepieces) > 1:                                               # plusieurs « pièces uniques » sur une photo = morceaux d'un même vêtement
            keep = max(onepieces, key=lambda t: t[1])[0]
            for key, _ in onepieces:
                if key == keep: continue
                e = report[key]; _discard(store, e["id"]); report[key] = dict(e, status="discarded", json=None, discard_reason="duplicate", reasons=e["reasons"] + [f"duplicate_of:{report[keep]['id']}"])
                if verbose: print(f"{key:34s} → ignorée (doublon de {report[keep]['id']})")
    rep_path.write_text(json.dumps(dict(version="itemcard_v1", items=list(report.values())), indent=2, ensure_ascii=False), encoding="utf-8")
    cards = [json.loads(Path(store / e["json"]).read_text(encoding="utf-8")) for e in report.values() if e["json"] and (store / e["json"]).exists()]
    (store / "wardrobe.json").write_text(json.dumps(cards, indent=2, ensure_ascii=False), encoding="utf-8")
    kept = [e for e in report.values() if e["status"] != "discarded"]
    summary = dict(photos=len(images), cards=len(cards), ignorees=sum(e["status"] == "discarded" for e in report.values()),
                   needs_review=sum(e["status"] == "needs_review" for e in kept), unsupported=sum(e["status"] == "unsupported" for e in kept),
                   wardrobe_json=str(store / "wardrobe.json"))
    if verbose: print(summary)
    return summary


# ---------------------------------------------------------------- explication du rapport
REASON_HELP = {
    "warmth_formality_placeholder": ("INFO", "warmth/formality : valeur neutre provisoire (le dictionnaire d'Amir n'est pas chargé). Ne déclenche PAS needs_review."),
    "slot_mapping_unconfirmed": ("REVUE (voulue)", "pièce traditionnelle : son slot (onepiece) n'est pas encore validé avec Amir -> toujours needs_review. Désactivable : CFG.confirm_traditional = False."),
    "traditional_subtype_from_neighbours": ("INFO", "sous-type (jebba, farmla…) lu chez les voisins tunisiens."),
    "traditional_subtype_unknown": ("INFO", "pas assez de voisins tunisiens avec un sous-type pour en annoncer un."),
    "slot_disagrees_with_segmentation": ("REVUE", "grande pièce : la segmentation dit A, la reconnaissance dit B, et ce n'est pas un cas normal (voir recognition/outfit.py)."),
    "crop": ("REVUE", "qualité du crop : garment_cut_by_frame = vêtement coupé par le bord de la photo ; no_garment_segmented = rien de découpé ; low_resolution = trop petit."),
    "low_confidence": ("REVUE", "confiance slot/type sous CFG.review_conf (0,60)."),
    "agent_could_not_decide": ("REVUE", "ni le LLM ni le kNN n'ont donné d'avis fiable (route REVIEW de l'agent)."),
    "agent_review_used_knn_top": ("INFO", "la fiche utilise la meilleure catégorie du kNN faute de décision de l'agent."),
    "colour_from_pixels_or_default": ("INFO", "couleur inconnue du LLM : prise dans les pixels."),
    "usage_home_mapped_to_Casual": ("INFO", "usage 'home' converti en 'Casual' (valeur non prévue par le contrat)."),
    "duplicate_of": ("IGNORÉE", "autre morceau du même vêtement dans la même photo : une seule fiche est gardée."),
    "fragment_of_other_garment": ("IGNORÉE", "petite pièce (chaussures/sac/accessoire) reconnue comme un autre vêtement : fragment, pas un article."),
    "no_slot_for_category": ("NON SUPPORTÉE", "catégorie sans slot (ex. underwear) : aucune fiche."),
}


def explain_report(store: str | Path, top: int = 12) -> dict:
    """Lit recognition_report.json et explique, chiffres à l'appui, POURQUOI des fiches sont en needs_review ou ignorées."""
    items = json.loads((Path(store) / "recognition_report.json").read_text(encoding="utf-8"))["items"]
    st = Counter(e["status"] for e in items); photos = Counter(e["source_file"] for e in items)
    print(f"{len(items)} pièces découpées sur {len(photos)} photos | statuts : {dict(st)}")
    print("pièces par photo :", dict(sorted(Counter(photos.values()).items())))
    kept = [e for e in items if e["status"] != "discarded"]
    base = lambda r: r.split(":")[0]
    rc = Counter(base(r) for e in kept for r in e["reasons"])
    print("\nRAISONS (nombre de fiches concernées) :")
    for k, v in rc.most_common(top): print(f"  {v:4d}  {k:40s} [{REASON_HELP.get(k, ('?', ''))[0]}] {REASON_HELP.get(k, ('', ''))[1]}")
    dis = Counter(r.split(":", 1)[1] for e in kept for r in e["reasons"] if r.startswith("slot_disagrees"))
    if dis: print("\nDésaccords segmentation → reconnaissance :", dict(dis.most_common(8)))
    blocking = {"slot_mapping_unconfirmed", "slot_disagrees_with_segmentation", "crop", "low_confidence", "agent_could_not_decide"}
    nr = [e for e in kept if e["status"] == "needs_review"]; only_trad = [e for e in nr if {base(r) for r in e["reasons"]} & blocking <= {"slot_mapping_unconfirmed"}]
    print(f"\nneeds_review : {len(nr)} fiche(s), dont {len(only_trad)} UNIQUEMENT parce que le slot des pièces traditionnelles n'est pas validé (revue voulue).")
    disc = Counter(e.get("discard_reason") for e in items if e["status"] == "discarded")
    if disc: print("ignorées :", dict(disc))
    return dict(statuts=dict(st), raisons=dict(rc), ignorees=dict(disc))

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/recognition/mock.py
"""Faux module de reconnaissance (livrable « mock ») : renvoie des fiches VALIDES sans charger aucun modèle.
Déterministe (dépend du nom de fichier). Les autres objectifs l'utilisent pour tester leurs branchements."""
from __future__ import annotations
import hashlib
from pathlib import Path
import numpy as np
from contracts import RecognitionResult, ItemCard, Confidence, slot_of, default_type, warmth_formality

_POOL = [("jacket", "blue", "Casual"), ("jeans", "blue", "Casual"), ("tshirt", "white", "Casual"), ("dress", "black", "Party"),
         ("shoes", "black", "Casual"), ("shirt", "white", "Formal"), ("bag", "brown", "Casual"), ("skirt", "red", "Party"),
         ("coat", "grey", "Formal"), ("accessory", "black", "Casual")]


class MockRecognizer:
    def recognize(self, crop_path, item_id=None, raw_path=None, out_dir=None, mode=None) -> RecognitionResult:
        h = int(hashlib.md5(str(Path(crop_path).name).encode()).hexdigest(), 16); cat, col, usage = _POOL[h % len(_POOL)]
        item_id = item_id or f"it_{h % 10**6:06d}"; w, f, _ = warmth_formality(cat)
        emb_rel = f"emb/{item_id}.npy"
        if out_dir:
            rng = np.random.default_rng(h % 2**32); v = rng.normal(size=512).astype("float32"); v /= np.linalg.norm(v)
            (Path(out_dir) / "emb").mkdir(parents=True, exist_ok=True); np.save(Path(out_dir) / emb_rel, v)
        card = ItemCard(id=item_id, name=f"{col.title()} {cat}", slot=slot_of(cat), type=default_type(cat), colour=[col], usage=usage, warmth=w,
                        formality=f, gender="Unisex", image=f"items/{item_id}_raw.jpg", image_crop=f"items/{item_id}_crop.png", embedding=emb_rel,
                        confidence=Confidence(slot=0.9, colour=0.7, type=0.8), source="obj1.llm")
        return RecognitionResult(status="ok", card=card, category=cat, reasons=["MOCK"], trace=["mock"], debug={"mock": True})

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/evaluation/__init__.py
# eval package

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/evaluation/metrics.py
from __future__ import annotations
from math import comb
import numpy as np


def wilson(k: int, n: int, z: float = 1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z ** 2 / n; c = (p + z ** 2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return float(c - h), float(c + h)


def macro_f1(y_true, y_pred) -> float:
    labels = sorted(set(y_true) | set(y_pred)); fs = []
    for c in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c); fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        fs.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(fs)) if fs else float("nan")


def bootstrap_ci(y_true, y_pred, stat, n_boot: int = 2000, seed: int = 0):
    rng = np.random.default_rng(seed); yt, yp = np.asarray(y_true), np.asarray(y_pred); n = len(yt); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n); vals.append(stat(yt[i].tolist(), yp[i].tolist()))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def mcnemar_exact(correct_a, correct_b) -> float:
    """p-valeur bilatérale exacte (binomiale) entre deux systèmes évalués sur les mêmes images."""
    b = sum(1 for x, y in zip(correct_a, correct_b) if x and not y); c = sum(1 for x, y in zip(correct_a, correct_b) if y and not x); n = b + c
    if n == 0: return 1.0
    k = min(b, c); return float(min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n))


def confidence_table(conf, correct, bins=(0, 0.6, 0.7, 0.85, 1.0001)):
    conf, correct = np.asarray(conf, float), np.asarray(correct, bool); rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi); rows.append(dict(tranche=f"[{lo:.2f};{min(hi, 1):.2f}]", n=int(m.sum()), precision=float(correct[m].mean()) if m.any() else float("nan")))
    return rows

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/evaluation/eval_catalogue.py
"""Section B du notebook : évaluation sur les 120 images de catalogue réservées + ablations.
Sert de jeu DEV pour régler les seuils (jamais les 30 photos réelles). Les appels LLM sont en cache : les ablations sont quasi gratuites.

    from evaluation.eval_catalogue import run_catalogue_eval
    run_catalogue_eval(svc, tag="v1")                                      # full
    run_catalogue_eval(svc, tag="llm_only", mode="llm_only")
    run_catalogue_eval(svc, tag="knn_only", mode="knn_only")
    svc.cfg.after_rag_policy = "llm"; run_catalogue_eval(svc, tag="after_rag_llm")"""
from __future__ import annotations
import json
import pandas as pd
from recognition.imaging import load_image
from recognition.taxonomy import coarse
from .metrics import wilson


def run_catalogue_eval(svc, tag: str = "v1", mode: str = "full") -> pd.DataFrame:
    out = svc.cfg.out / f"agent_eval_{tag}.jsonl"; done = {}
    if out.exists():
        for line in open(out, encoding="utf-8"):
            try: r = json.loads(line); done[r["id"]] = r
            except Exception: pass
    for _, row in svc.test_df.iterrows():
        if int(row.id) in done: continue
        r = svc.agent.run(load_image(svc.img_src / f"{int(row.id)}.jpg"), mode)
        rec = dict(id=int(row.id), truth=row.category, truth_color=row.color if isinstance(row.color, str) else None,
                   truth_gender=row.gender if isinstance(row.gender, str) else None, final=r["category"], source=r["source"], status=r["status"],
                   confidence=r["confidence"], H=r["H"], calls=r["llm_calls"], latency=r["latency_s"], color=r["color"], gender=r["gender"],
                   fp_valid=r["first"]["valid"], fp_cat=r["first"]["category"], fp_color=r["first"]["color"], fp_gender=r["first"]["gender"], trace="\n".join(r["trace"]))
        with open(out, "a", encoding="utf-8") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        done[rec["id"]] = rec
    ev = pd.DataFrame(done.values()); ev = ev[ev.id.isin(svc.test_df.id)].reset_index(drop=True)
    ok = ev.final == ev.truth; lo, hi = wilson(int(ok.sum()), len(ev)); ans = ev.status == "OK"
    print(f"[{tag}] précision finale {ok.mean():.3f} IC95 [{lo:.3f};{hi:.3f}] | groupée {sum(coarse(a) == coarse(b) for a, b in zip(ev.final, ev.truth)) / len(ev):.3f} "
          f"| REVIEW {1 - ans.mean():.3f} | appels LLM {ev.calls.mean():.2f}")
    print(ev.assign(correct=ok).groupby("source").agg(n=("id", "count"), précision=("correct", "mean")).round(3))
    tc = ev.truth_color.notna(); tg = ev.truth_gender.notna()
    print(f"couleur finale {(ev.color[tc] == ev.truth_color[tc]).mean():.3f} | genre final {(ev.gender[tg] == ev.truth_gender[tg]).mean():.3f}")
    ev.to_csv(svc.cfg.out / f"agent_eval_{tag}.csv", index=False); return ev

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/evaluation/eval_real.py
"""Évaluation sur les PHOTOS RÉELLES (livrable : 30 photos de téléphone, déjà découpées par le crop partagé).

labels.csv (étiqueté par 2 personnes puis arbitré) — colonnes :
    image_path, true_category (17 classes), true_colour (14 couleurs)   [facultatif : true_type, true_usage, wardrobe]
    python -m evaluation.eval_real labels.csv      (ou run_real_eval(svc, 'labels.csv') depuis le notebook)

Compare 3 systèmes sur les MÊMES images : agent complet, LLM seul, kNN seul (références exigées par l'évaluation commune)."""
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
from contracts import slot_of
from .metrics import wilson, macro_f1, bootstrap_ci, mcnemar_exact, confidence_table

MODES = ("full", "llm_only", "knn_only")


def run_real_eval(svc, labels_csv: str | Path, out_dir: str | Path | None = None, modes=MODES) -> dict:
    lab = pd.read_csv(labels_csv).fillna(""); base = Path(labels_csv).parent; out_dir = Path(out_dir or svc.cfg.out / "real_eval"); out_dir.mkdir(parents=True, exist_ok=True)
    allrows, summ = {}, []
    for m in modes:
        rows = []
        for _, r in lab.iterrows():
            p = Path(r.image_path); p = p if p.is_absolute() else base / p
            res = svc.recognize(p, item_id=None, out_dir=out_dir / f"cards_{m}", mode=m); c = res.card
            rows.append(dict(image=str(p), true_category=r.true_category, pred_category=res.category, true_slot=slot_of(r.true_category) or "none",
                             pred_slot=c.slot if c else "none", true_colour=r.true_colour, pred_colour=(c.colour[0] if c else None),
                             true_type=r.get("true_type", ""), pred_type=(c.type if c else None), status=res.status, conf_slot=c.confidence.slot if c else 0.0,
                             conf_type=c.confidence.type if c else 0.0, conf_colour=c.confidence.colour if c else 0.0, source=res.debug.get("source", ""),
                             latency=res.debug.get("latency_s", 0.0), reasons=";".join(res.reasons)))
        df = pd.DataFrame(rows); df.to_csv(out_dir / f"predictions_{m}.csv", index=False); allrows[m] = df; n = len(df)
        ok_slot, ok_cat, ok_col = df.true_slot == df.pred_slot, df.true_category == df.pred_category, df.true_colour == df.pred_colour
        row = dict(systeme=m, n=n, slot=ok_slot.mean(), slot_ic=wilson(int(ok_slot.sum()), n), categorie=ok_cat.mean(), categorie_ic=wilson(int(ok_cat.sum()), n),
                   couleur=ok_col.mean(), couleur_ic=wilson(int(ok_col.sum()), n), f1_macro_cat=macro_f1(df.true_category.tolist(), df.pred_category.tolist()),
                   f1_macro_ic=bootstrap_ci(df.true_category.tolist(), df.pred_category.tolist(), macro_f1),
                   taux_revue=(df.status != "ok").mean())
        if (df.true_type != "").all(): ok_t = df.true_type == df.pred_type; row.update(type=ok_t.mean(), type_ic=wilson(int(ok_t.sum()), n))
        summ.append(row)
    S = pd.DataFrame(summ).set_index("systeme"); S.to_csv(out_dir / "summary.csv"); print(S.round(3).to_string())
    if "full" in allrows and "llm_only" in allrows:
        a, b = allrows["full"], allrows["llm_only"]
        print("McNemar exact (catégorie, full vs llm_only) p =", round(mcnemar_exact((a.true_category == a.pred_category).tolist(), (b.true_category == b.pred_category).tolist()), 4))
    if "full" in allrows:
        f = allrows["full"]; print("\nPrécision du slot par tranche de confiance (calibration) :")
        print(pd.DataFrame(confidence_table(f.conf_slot, f.true_slot == f.pred_slot)).round(3).to_string(index=False))
        err = f[f.true_category != f.pred_category]; print("\nConfusions :"); print(err.groupby(["true_category", "pred_category"]).size().sort_values(ascending=False).head(10).to_string())
    return dict(summary=S, predictions=allrows)


if __name__ == "__main__":
    from recognition.service import RecognitionService
    run_real_eval(RecognitionService.from_config(), sys.argv[1])

In [ ]:
%%writefile /content/drive/MyDrive/dressme_obj1/evaluation/eval_tunisian.py
"""Évaluation sur le jeu tunisien (split dev pour régler, split test UNE seule fois à la fin).

Compare, sur les MÊMES photos :
  clip_zeroshot            FashionCLIP seul, sans aucun exemple (sondes texte)  -> « FashionCLIP reconnaît-il ces pièces ? »
  knn_catalogue            kNN avec la base catalogue seule
  knn_+tn_refs             kNN avec les références tunisiennes (train) ajoutées
  llm_only                 Qwen seul
  full_catalogue           agent complet, base catalogue seule
  full_+tn_refs            agent complet, avec références tunisiennes
  full_+tn_refs_+prompt    idem + description des pièces tunisiennes dans le prompt
Mesures : précision (Wilson 95 %), F1 macro, rappel de `traditional_garment`, précision du sous-type (jebba, farmla…)."""
from __future__ import annotations
from pathlib import Path
import numpy as np, pandas as pd
from recognition.imaging import load_image, open_rgb
from recognition.itemcard_builder import traditional_subtype
from recognition.taxonomy import CATEGORIES
from recognition.tunisian import load_tunisian, prepare_crops
from .metrics import wilson, macro_f1

ZS_PROMPTS = {"tshirt": "a t-shirt", "shirt": "a button-up shirt", "top": "a blouse or tank top", "sweater": "a sweater or hoodie", "jacket": "a jacket",
              "blazer": "a blazer", "coat": "a long coat", "dress": "a dress", "skirt": "a skirt", "jeans": "a pair of jeans", "pants": "a pair of trousers",
              "shorts": "a pair of shorts", "underwear": "underwear", "shoes": "a pair of shoes", "bag": "a bag", "accessory": "a fashion accessory",
              "traditional_garment": "a traditional Tunisian garment such as a jebba, farmla, blouza or kaftan"}


def _zeroshot(svc, imgs):
    T = svc.shared.texts([f"a photo of {ZS_PROMPTS[c]}" for c in CATEGORIES]); I = svc.shared.images(imgs)
    return [CATEGORIES[i] for i in (I @ T.T).argmax(1)]


def run_tunisian_eval(svc, split: str = "dev", out_dir: str | Path | None = None, systems: tuple | None = None, tag: str = "") -> dict:
    cfg, agent = svc.cfg, svc.agent
    df, issues = load_tunisian(cfg, split)
    if not len(df): print(f"Aucune photo tunisienne valide dans le split '{split}'."); return {}
    df = prepare_crops(cfg, df, svc.cropper); imgs = [load_image(p) for p in df.crop_path]; out_dir = Path(out_dir or cfg.out / f"tunisian_eval_{split}{'_' + tag if tag else ''}"); out_dir.mkdir(parents=True, exist_ok=True)
    has_refs = getattr(svc, "retr_ref", None) is not getattr(svc, "retr_base", None)
    plan = {"clip_zeroshot": None, "knn_catalogue": ("knn_only", "base", False), "knn_+tn_refs": ("knn_only", "ref", False), "llm_only": ("llm_only", "base", False),
            "full_catalogue": ("full", "base", False), "full_+tn_refs": ("full", "ref", False), "full_+tn_refs_+prompt": ("full", "ref", True)}
    if not has_refs: plan = {k: v for k, v in plan.items() if "tn_refs" not in k}
    if systems: plan = {k: v for k, v in plan.items() if k in systems}
    keep = (agent.retriever, cfg.tunisian_prompt); preds, subs = {}, {}
    try:
        for name, spec in plan.items():
            if spec is None: preds[name] = _zeroshot(svc, imgs); continue
            mode, which, prompt = spec; agent.retriever = svc.retr_ref if which == "ref" else svc.retr_base; cfg.tunisian_prompt = prompt; P, S = [], []
            for im in imgs:
                r = agent.run(im, mode); P.append(r["category"] if r["category"] != "unknown" else r["knn"]["category"]); S.append(traditional_subtype(r["neighbors"], cfg.subtype_min_share)[0])
            preds[name] = P; subs[name] = S
    finally: agent.retriever, cfg.tunisian_prompt = keep
    truth = df.category.tolist(); rows = []
    for name, P in preds.items():
        ok = np.array([a == b for a, b in zip(truth, P)]); lo, hi = wilson(int(ok.sum()), len(ok)); trad = np.array([t == "traditional_garment" for t in truth])
        row = dict(systeme=name, n=len(ok), precision=ok.mean(), ic95=(round(lo, 3), round(hi, 3)), f1_macro=macro_f1(truth, P),
                   rappel_traditional=(ok[trad].mean() if trad.any() else np.nan), n_traditional=int(trad.sum()))
        if name in subs:
            m = np.array([t == "traditional_garment" and s != "" for t, s in zip(truth, df.subtype)])
            row["precision_sous_type"] = float(np.mean([a == b for a, b, k in zip(df.subtype, subs[name], m) if k])) if m.any() else np.nan
        rows.append(row)
    S = pd.DataFrame(rows).set_index("systeme"); S.to_csv(out_dir / "summary.csv")
    pd.DataFrame({"image": df.image_path, "truth": truth, **{k: v for k, v in preds.items()}}).to_csv(out_dir / "predictions.csv", index=False)
    print(S.round(3).to_string()); print(f"\nrésultats : {out_dir}  (split={split}, n={len(df)})")
    if split == "test": print("⚠️ split test : à ne regarder qu'UNE fois, après avoir figé les réglages sur dev.")
    return dict(summary=S, predictions=preds, data=df)

## 4. Configuration

In [ ]:
import os, sys
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
os.environ['DRESSME_ROOT'] = str(DATA_ROOT)
# os.environ['DRESSME_TYPE_DICT']    = str(DATA_ROOT / 'obj6_type_dictionary.json')   # dictionnaire warmth/formality d'Amir
# os.environ['DRESSME_SHARED_EMBED'] = 'common.embed:embed_image'                      # script FashionCLIP commun de Suzana
# os.environ['DRESSME_CROP_HOOK']    = 'segment_amir:crop_photo'                       # crop d'Amir (remplace mon crop de repli)
from recognition.config import CFG
from contracts import load_type_dictionary
CFG.root = DATA_ROOT
CFG.tn_refs = True               # références tunisiennes (train) dans le kNN/RAG
CFG.tunisian_prompt = False      # True = description des pièces tunisiennes dans le prompt (à comparer sur DEV)
# CFG.after_rag_policy = 'knn'   # mesuré sur le catalogue : 'llm' n'a PAS amélioré (0,908 vs 0,917)
FINETUNED_ACTIVE = False
print('dictionnaire officiel chargé :', load_type_dictionary(), '| données :', CFG.root)
if not TRAINING_READY.exists(): print('ℹ️ dressme_training_ready.csv absent :', TRAINING_READY, '(utile seulement pour construire labels.csv à l\'étape 5)')

## 5. Construire `tunisian_dataset/labels.csv` depuis `dressme_training_ready.csv`
⚠️ Dans ton fichier, `origin = tunisia_auto_qwen` : si ces étiquettes viennent de Qwen lui-même, elles ne peuvent pas servir de vérité pour **évaluer** Qwen (évaluation circulaire). Fais vérifier à la main au moins les splits `dev` et `test`, et donne un `group_id` commun aux photos d'une même séance/tenue.

Si `labels.csv` existe déjà, il est **conservé** (tes corrections manuelles ne sont pas écrasées). `REBUILD_LABELS = True` le reconstruit (sauvegarde `labels_backup.csv`).

In [ ]:
import shutil
import pandas as pd

REBUILD_LABELS = False     # True = reconstruit labels.csv (ÉCRASE les corrections manuelles ; une sauvegarde est faite)

if TN_LABELS.exists() and not REBUILD_LABELS:
    print('labels.csv existe déjà : conservé ->', TN_LABELS, '(REBUILD_LABELS=False)')
else:
    if TN_LABELS.exists(): shutil.copy2(TN_LABELS, TN_LABELS.with_name('labels_backup.csv')); print('sauvegarde : labels_backup.csv')

    if not TRAINING_READY.exists():
        raise FileNotFoundError(
            f"Fichier absent : {TRAINING_READY}\n"
            "Copie dressme_training_ready.csv dans data/database/."
        )

    ready = pd.read_csv(TRAINING_READY, low_memory=False)

    if "is_tunisian" not in ready.columns:
        raise RuntimeError(
            "La colonne 'is_tunisian' manque dans dressme_training_ready.csv."
        )

    tn = ready[
        ready["is_tunisian"].fillna(0).astype(int).eq(1)
    ].copy()

    if "usable_for_training" in tn.columns:
        tn = tn[
            tn["usable_for_training"].fillna(0).astype(int).eq(1)
        ].copy()

    if "needs_review" in tn.columns:
        tn = tn[
            tn["needs_review"].fillna(1).astype(int).eq(0)
        ].copy()

    print("Lignes tunisiennes acceptées :", len(tn))

    VALID_CATEGORIES = {
        "tshirt", "shirt", "top", "sweater",
        "jacket", "blazer", "coat", "dress",
        "skirt", "jeans", "pants", "shorts",
        "underwear", "shoes", "bag", "accessory",
        "traditional_garment",
    }

    VALID_COLORS = {
        "beige", "black", "blue", "brown", "green", "grey",
        "metallic", "multicolor", "orange", "pink", "purple",
        "red", "white", "yellow",
    }

    VALID_GENDERS = {"male", "female", "unisex"}
    VALID_SEASONS = {"summer", "winter", "spring", "fall", "all_season"}
    VALID_OCCASIONS = {"casual", "formal", "sport", "party", "traditional", "home"}

    def clean(v):
        if pd.isna(v):
            return ""
        return str(v).strip().lower()

    def norm_category(v):
        v = clean(v)
        aliases = {
            "t-shirt": "tshirt",
            "t_shirt": "tshirt",
            "tee": "tshirt",
            "trousers": "pants",
            "pant": "pants",
            "hoodie": "sweater",
            "sweatshirt": "sweater",
            "blouse": "top",
            "traditional": "traditional_garment",
            "traditional garment": "traditional_garment",
        }
        return aliases.get(v, v)

    def norm_color(v):
        v = clean(v).replace(" ", "_")
        aliases = {
            "gray": "grey",
            "silver": "metallic",
            "gold": "metallic",
            "multi": "multicolor",
            "multi_color": "multicolor",
            "multicolored": "multicolor",
            "navy": "blue",
            "navy_blue": "blue",
            "cream": "beige",
            "unknown": "",
        }
        v = aliases.get(v, v)
        return v if v in VALID_COLORS else ""

    def norm_gender(v):
        v = clean(v)
        aliases = {
            "men": "male",
            "man": "male",
            "women": "female",
            "woman": "female",
            "unknown": "",
        }
        v = aliases.get(v, v)
        return v if v in VALID_GENDERS else ""

    def norm_season(v):
        v = clean(v)
        aliases = {
            "autumn": "fall",
            "all": "all_season",
            "all_seasons": "all_season",
            "unknown": "",
        }
        v = aliases.get(v, v)
        return v if v in VALID_SEASONS else ""

    def norm_occasion(v):
        v = clean(v).replace(" ", "_")
        aliases = {
            "sports": "sport",
            "smart_casual": "casual",
            "everyday": "casual",
            "ethnic": "traditional",
            "ceremonial": "traditional",
            "ceremony": "traditional",
            "unknown": "",
        }
        v = aliases.get(v, v)
        return v if v in VALID_OCCASIONS else ""

    def infer_subtype(row, category):
        if category != "traditional_garment":
            return ""
        garment = clean(row.get("garment_name", ""))
        for s in ("jebba", "farmla", "blouza", "kaftan", "fouta"):
            if s in garment:
                return s
        return "other"

    def find_source_image(row):
        record_id = str(row.get("record_id", "")).strip()

        # 1. Chemin déjà présent dans le CSV, si encore valide
        old = row.get("image_path", "")
        if isinstance(old, str) and old.strip():
            p = Path(old)
            if p.is_file():
                return p

        # 2. data/tunisian/images/<record_id>.<ext>
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = TN_SOURCE_IMAGES / f"{record_id}{ext}"
            if p.is_file():
                return p

        return None

    rows = []
    missing = []
    invalid = []

    for _, r in tn.iterrows():
        record_id = str(r.get("record_id", "")).strip()
        category = norm_category(r.get("category", ""))

        if category not in VALID_CATEGORIES:
            invalid.append((record_id, category))
            continue

        src = find_source_image(r)

        if src is None:
            missing.append(record_id)
            continue

        dst = TN_DATASET_IMAGES / f"{record_id}{src.suffix.lower()}"

        if not dst.exists():
            shutil.copy2(src, dst)

        rows.append({
            "image_path": (Path("images") / dst.name).as_posix(),
            "category": category,
            "subtype": infer_subtype(r, category),
            "colour": norm_color(r.get("color", "")),
            "gender": norm_gender(r.get("gender", "")),
            "season": norm_season(r.get("season", "")),
            "occasion": norm_occasion(r.get("occasion", "")),
            "event": "",
            "origin": "tunisia_auto_qwen",
            "group_id": f"tn_{record_id}",
            "split": "",      # attribution automatique 60/20/20 par recognition/tunisian.py
            "cropped": "0",
        })

    labels = pd.DataFrame(rows)
    labels.to_csv(TN_LABELS, index=False, encoding="utf-8-sig")

    print("labels.csv créé :", TN_LABELS)
    print("Lignes valides  :", len(labels))
    print("Images absentes :", len(missing), missing[:10])
    print("Catégories invalides :", len(invalid), invalid[:10])

    display(labels.head(20))


## 6. Contrôle des données tunisiennes

In [ ]:
from recognition.tunisian import report, load_tunisian
tab = report(CFG)       # erreurs, avertissements, comptage par classe et par split
tn_all, issues = load_tunisian(CFG)
print('\nTotal lignes tunisiennes valides :', len(tn_all))
if len(tn_all): print(tn_all['split'].value_counts().to_string())

## 7. Vérifier / réparer les images illisibles
Le projet ignore de toute façon les images illisibles ; cette cellule les liste et tente de les recopier depuis `data/tunisian/images/`.

In [ ]:
import shutil, pandas as pd
from recognition.tunisian import readable
REMOVE_BAD = False        # True : retire de labels.csv les lignes dont l'image reste illisible (sauvegarde labels_backup.csv)
labels = pd.read_csv(TN_LABELS, encoding='utf-8-sig'); bad, repaired = [], []
for idx, row in labels.iterrows():
    prepared = TN_DATASET_DIR / str(row['image_path']); ok, why = readable(prepared)
    if ok: continue
    orig = TN_SOURCE_IMAGES / prepared.name; ok2, why2 = readable(orig) if orig.exists() else (False, 'original absent')
    print(f'⚠️ {prepared.name} — {why}', end=' ')
    if ok2: shutil.copy2(orig, prepared); repaired.append(prepared.name); print("→ recopiée depuis l'original ✅")
    else: bad.append(idx); print(f'→ original aussi inutilisable ({why2}) ❌')
print(f'\nImages vérifiées : {len(labels)} | réparées : {len(repaired)} | irréparables : {len(bad)}')
if bad and REMOVE_BAD:
    labels.to_csv(TN_DATASET_DIR / 'labels_backup.csv', index=False, encoding='utf-8-sig')
    labels.drop(index=bad).reset_index(drop=True).to_csv(TN_LABELS, index=False, encoding='utf-8-sig'); print('labels.csv mis à jour')

## 8. Chargement de l'agent
Qwen2.5-VL (4 bits) + FashionCLIP gelé + base kNN catalogue + références tunisiennes `train` (sans entraînement). Les crops tunisiens sont créés et mis en cache dans `tunisian_dataset/crops/`.

In [ ]:
from recognition.service import RecognitionService
svc = RecognitionService.from_config(CFG)
print('✅ prêt | base catalogue :', len(svc.kb_df), '| références tunisiennes :', svc.n_tn_refs, '| test catalogue :', len(svc.test_df))

## 9. Évaluations de référence (encodeur gelé)
**9a. Catalogue (120 images)** : résultats v1 connus = 0,917 (agent complet) · 0,833 (Qwen seul) · 0,892 (kNN seul).

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
from evaluation.eval_catalogue import run_catalogue_eval
run_catalogue_eval(svc, 'v1'); run_catalogue_eval(svc, 'llm_only', 'llm_only'); run_catalogue_eval(svc, 'knn_only', 'knn_only')

**9b. Jeu tunisien, split `dev`** : FashionCLIP zéro-shot, kNN avec/sans références tunisiennes, Qwen seul, agent complet, avec/sans prompt tunisien. C'est ici qu'on voit si les photos tunisiennes aident. `dev` sert à régler ; `test` : une seule fois, à la fin.

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
from evaluation.eval_tunisian import run_tunisian_eval
out_dev = run_tunisian_eval(svc, 'dev')

## 10. Fine-tuning FashionCLIP (optionnel)
À lancer **après** 9b, pour comparer. On entraîne une **copie** sur `train` (catalogue + tunisien) : l'embedding des fiches reste celui du FashionCLIP gelé, commun à l'équipe ; l'encodeur affiné ne sert qu'au kNN interne.
`RUN_FINETUNE = True` pour lancer.

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
RUN_FINETUNE = False
if RUN_FINETUNE:
    FINETUNED_ACTIVE = svc.enable_finetuned()
    if FINETUNED_ACTIVE: out_ft = run_tunisian_eval(svc, 'dev', tag='finetuned')     # à comparer avec la table 9b
print('FashionCLIP fine-tuné actif :', FINETUNED_ACTIVE)

## 11. PHOTO → crop → fichier JSON
Mets tes photos dans `data/inbox/` (ou téléverse-les ci-dessous). Chaque pièce détectée donne `store/items/it_XXXX.json` + `store/emb/it_XXXX.npy` ; le récapitulatif est `store/wardrobe.json`.
`crop=True` (auto) : personne visible → SegFormer, un crop par pièce ; sinon rembg. `crop='flat'` / `'worn'` pour forcer ; `crop=False` si les images sont déjà des crops.

In [ ]:
# (facultatif) téléverser des photos directement dans inbox/
from google.colab import files
up = files.upload()
for fn, data in up.items(): (INBOX / fn).write_bytes(data)
print(len(up), 'photo(s) dans', INBOX)

**(Facultatif) Nettoyer le store sans relancer la reconnaissance.** `FRESH = True` dans la cellule suivante le fait déjà. Ici : `RUN_CLEAN = True` pour vider `items/`, `emb/`, `wardrobe.json`, `recognition_report.json` (`agent_final/` et `tunisian_dataset/` ne sont jamais touchés) ; `ARCHIVE = True` déplace l'ancien contenu dans `store_archive_<date>` au lieu de le supprimer.

In [ ]:
from recognition.batch import reset_store
RUN_CLEAN = False      # sécurité : ne rien supprimer tant que tu ne passes pas ceci à True
ARCHIVE = False
if RUN_CLEAN: print(reset_store(STORE, archive=ARCHIVE))
else: print('Rien supprimé (RUN_CLEAN = False).')

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
from recognition.batch import recognize_folder, reset_store, explain_report

# --- réglages du crop (relus avant chaque lot) ---
CFG.crop_strategy = 'outfit'      # 'main' = UNE fiche par photo (la pièce la plus grande) : conseillé pour des photos d'un seul vêtement (ex. jeu tunisien)
CFG.crop_accessories = True       # False = ignorer chapeau / écharpe / ceinture
CFG.confirm_traditional = True    # False = ne plus forcer needs_review sur les pièces traditionnelles (seulement APRÈS validation du slot avec Amir)

FRESH = True     # True = repart d'un store propre (ids it_0001…) ; False = ajoute à l'existant (ids stables)
LIMIT = 10       # démo : seulement les 10 premières images de inbox/ ; None = toutes
summary = recognize_folder(svc, INBOX, STORE, crop=True, fresh=FRESH, limit=LIMIT)

## 12. Où sont mes résultats JSON ?

In [ ]:
import json, glob
print('Dossier :', STORE)
for f in sorted(glob.glob(str(STORE / 'items' / 'it_*.json')))[:50]: print('  fiche    ', Path(f).relative_to(STORE))
for f in sorted(glob.glob(str(STORE / 'emb' / '*.npy')))[:5]: print('  embedding', Path(f).relative_to(STORE))
print('  récapitulatif :', STORE / 'wardrobe.json', '| rapport :', STORE / 'recognition_report.json')
w = json.load(open(STORE / 'wardrobe.json')); print(len(w), 'fiche(s) ; première :'); print(json.dumps(w[0], indent=2, ensure_ascii=False) if w else '—')
rep = json.load(open(STORE / 'recognition_report.json'))['items']
print("\nÀ confirmer par l'utilisateur :")
for e in rep:
    if e['status'] == 'needs_review': print(' ', e['id'], e['status'], e['reasons'])

**Pourquoi ces avertissements ?** La cellule suivante lit `recognition_report.json` et explique, chiffres à l'appui, les raisons des `needs_review` et des pièces ignorées (fragments, doublons).

In [ ]:
from recognition.batch import explain_report
_ = explain_report(STORE)

## 13. (Optionnel) Un JSON par image du split `train` tunisien
Pour tester d'abord : `EXPORT_LIMIT = 5` ; pour tout exporter : `EXPORT_LIMIT = None`.

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
import json
from tqdm.auto import tqdm
from recognition.tunisian import load_tunisian, prepare_crops

EXPORT_LIMIT = 5
OVERWRITE = False      # True = régénère les JSON déjà présents (sinon ils sont simplement relus : relancer ne change rien)

train_df, issues = load_tunisian(CFG, "train")

if len(train_df) == 0:
    print("Aucune image tunisienne dans le split train.")
else:
    train_df = prepare_crops(
        CFG,
        train_df,
        svc.cropper,
    )

    if EXPORT_LIMIT is not None:
        train_df = train_df.head(EXPORT_LIMIT)

    run_name = "finetuned" if FINETUNED_ACTIVE else "frozen"
    run_dir = JSON_OUTPUT / run_name
    per_image_dir = run_dir / "per_image"
    result_store = run_dir / "runtime_store"

    per_image_dir.mkdir(parents=True, exist_ok=True)
    result_store.mkdir(parents=True, exist_ok=True)

    all_results = []

    for row in tqdm(
        train_df.itertuples(index=False),
        total=len(train_df),
        desc="Tunisian TRAIN -> JSON",
    ):
        item_id = Path(row.image_path).stem
        json_path = per_image_dir / f"{item_id}.json"

        if json_path.exists() and not OVERWRITE:
            payload = json.loads(json_path.read_text(encoding="utf-8"))
            all_results.append(payload)
            continue

        try:
            result = svc.recognize(
                row.crop_path,
                item_id=item_id,
                out_dir=result_store,
                mode="full",
            )

            payload = {
                "id": item_id,
                "input": {
                    "image_path": row.abs_path,
                    "crop_path": row.crop_path,
                    "dataset_label": {
                        "category": row.category,
                        "subtype": row.subtype,
                        "colour": row.colour,
                        "gender": row.gender,
                        "season": row.season,
                        "occasion": row.occasion,
                    },
                },
                "output": result.model_dump(mode="json"),
                "pipeline": {
                    "fashionclip_finetuned": bool(FINETUNED_ACTIVE),
                    "qwen_model": CFG.llm_id,
                    "fashionclip_model": CFG.embedder_id,
                },
            }

        except Exception as e:
            payload = {
                "id": item_id,
                "input": {
                    "image_path": row.abs_path,
                    "crop_path": row.crop_path,
                },
                "output": {
                    "status": "error",
                    "category": "unknown",
                    "error": repr(e),
                },
            }

        json_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8",
        )

        all_results.append(payload)

    global_json = run_dir / "all_training_results.json"

    global_json.write_text(
        json.dumps(all_results, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

    print("✅ JSON par image :", per_image_dir)
    print("✅ JSON global    :", global_json)
    print("Nombre exporté    :", len(all_results))


## 14. Photos réelles (livrable : ~30 photos de téléphone étiquetées)
`data/real30/labels.csv` : `image_path` (crops), `true_category` (17 classes), `true_colour` (14 couleurs). Ces photos ne doivent **pas** figurer dans `tunisian_dataset/` avec un split `train`.

In [ ]:
try:
    svc
except NameError:
    raise RuntimeError('Agent non chargé (runtime redémarré ?) : ré-exécute la cellule de configuration puis le chargement de l\'agent.')
from evaluation.eval_real import run_real_eval
REAL30_LABELS = REAL30 / 'labels.csv'
if REAL30_LABELS.exists(): out_real = run_real_eval(svc, REAL30_LABELS)
else: print('Pas encore de fichier :', REAL30_LABELS, '(étape optionnelle tant que real30/labels.csv n\'est pas prêt)')

## 15. Mock + schéma JSON (contrat gelé)

In [ ]:
from recognition.mock import MockRecognizer
from contracts import export_json_schema
print(MockRecognizer().recognize('x.png').card.model_dump_json(indent=2)[:400])
export_json_schema(PROJECT_ROOT / 'contracts' / 'schema' / 'itemcard_v1.schema.json'); print('Schéma écrit dans Drive')

## 16. Diagnostic : « tout s'exécute mais rien ne change »
Affiche (1) si le code chargé en mémoire est la dernière version, (2) les JSON réellement présents dans Drive avec leur date, (3) les dossiers d'entrée/sortie, (4) les paramètres actifs, (5) les caches qui expliquent qu'un résultat ne bouge pas.

In [ ]:
import time, glob, json
from pathlib import Path
age = lambda p: time.strftime('%d/%m %H:%M', time.localtime(Path(p).stat().st_mtime))
count = lambda pat: len(glob.glob(pat, recursive=True))

print('1) CODE EN MÉMOIRE = dernière version ?  (False => Exécution > Redémarrer la session, puis relancer 2 -> 8)')
import recognition.tunisian as _T, recognition.config as _C, recognition.service as _S, recognition.batch as _B
for name, ok in {'tunisian : lecture robuste des images': hasattr(_T, 'readable'), 'config : rembg_model': hasattr(_C.Config(), 'rembg_model'),
                 'service : enable_finetuned': hasattr(_S.RecognitionService, 'enable_finetuned'), 'batch : recognize_folder(crop=...)': 'crop' in _B.recognize_folder.__code__.co_varnames}.items():
    print('  ', '✅' if ok else '❌', name)

print('\n2) JSON PRÉSENTS DANS DRIVE (20 plus récents)')
js = sorted(glob.glob(str(DATA_ROOT / '**' / '*.json'), recursive=True), key=lambda p: Path(p).stat().st_mtime, reverse=True)
for p in js[:20]: print('  ', age(p), Path(p).relative_to(DATA_ROOT))
print('   total :', len(js), 'fichier(s) .json sous', DATA_ROOT)

print('\n3) DOSSIERS')
print('   inbox/ (entrée)         :', count(str(INBOX / '*')), 'fichier(s)  <- section 11 ne produit RIEN si vide')
print('   store/items/*.json      :', count(str(STORE / 'items' / 'it_*.json')), '| wardrobe.json :', age(STORE / 'wardrobe.json') if (STORE / 'wardrobe.json').exists() else 'absent')
print('   training_json_results   :', count(str(JSON_OUTPUT / '**' / 'per_image' / '*.json')), 'JSON par image')
print('   labels.csv              :', age(TN_LABELS) if TN_LABELS.exists() else 'absent', '(conservé tel quel si REBUILD_LABELS=False)')

print('\n4) PARAMÈTRES ACTIFS')
print('   tn_refs =', CFG.tn_refs, '| tunisian_prompt =', CFG.tunisian_prompt, '| after_rag_policy =', CFG.after_rag_policy, '| crop_mode =', CFG.crop_mode)
print('   fine-tuning actif :', FINETUNED_ACTIVE, '(RUN_FINETUNE=False par défaut)')
try: print('   références tunisiennes dans le kNN :', svc.n_tn_refs)
except NameError: print('   agent non chargé (svc absent)')

print('\n5) CACHES (ils expliquent qu\'un résultat identique revienne)')
af = CFG.out
for pat, why in (('llm_calls.jsonl', 'appels Qwen déjà faits (même image + même prompt = même réponse, réutilisée)'), ('agent_eval_*.jsonl', 'évaluation catalogue reprise là où elle s\'est arrêtée'), ('kb_meta.csv', 'base kNN catalogue')):
    for p in sorted(af.glob(pat)): print('  ', p.name.ljust(26), age(p), '-', why)
print('   crops tunisiens en cache :', count(str(TN_DATASET_DIR / 'crops' / '*.png')))
print('\nLecture : mêmes chiffres d\'évaluation = normal tant que rien ne change (décodage déterministe + cache). Pour voir un écart, change un paramètre (CFG.tunisian_prompt, RUN_FINETUNE, nouvelles photos train) puis relance 9b.')